# Stitch a production-grade pipeline from raw S3 landing to governed Redshift with full observability

You are the sole data engineer at a Series A startup. The CTO scheduled a Friday demo for investors. Your job tonight is to wire every piece from Labs 1 through 11 into a single working pipeline: Kinesis to Firehose, raw S3 to a Glue workflow with bookmarks, curated S3 to a Redshift COPY, CloudWatch alarms on row count and Glue job failure, EventBridge fan-out to one SNS topic, and Lake Formation grants scoped to the team_a analytics role. Then prove the cleanup script returns the account to zero accruing hourly cost within ten minutes.

This is the capstone. It is the most expensive lab in the cert. Plan for about $1 if you finish in 75 minutes and up to $1.50 if the 3-hour safety net is what cleans up. Cleanup is itself the graded final checkpoint: the validator waits 10 minutes after teardown and queries Cost Explorer.

The four deliverables map to four checkpoints:
1. Every critical resource running and registered: Redshift cluster `available`, Kinesis stream `ACTIVE`, safety-net Lambda + EventBridge schedule rule both `Enabled`, manifest has `critical=True` on all four, schedule expression fires at lab start plus 3 hours.
2. End-to-end canary. One tagged event (`event_id == 'lab12-canary'`) lands on Kinesis, travels through Firehose to raw S3, the Glue workflow processes it into curated Parquet, and Redshift `SELECT count(*) FROM curated_orders WHERE event_id = 'lab12-canary'` returns 1.
3. Alarm plus EventBridge plus SNS fires on the injected failure path. Broken batch produces row count 5; the row-count alarm trips to ALARM; SNS NumberOfMessagesPublished > 0; a separate Glue load job failure fans out through a second EB rule to the same SNS topic; the validator counts at least 2 distinct publishes.
4. Graded cleanup. `run_cleanup` returns `status='clean'`; 10 minutes later, Cost Explorer HOURLY granularity on the lab tag returns no new charges; tag audit shows zero orphans.

**Time.** About 120 minutes hands-on. The 180-minute session window accounts for the 4 to 6 minute cluster create, the Glue workflow runtime (about 15 minutes), and the 10-minute Cost Explorer wait.

**Cost.** About $0.75 to $1.50 per session. Redshift `dc2.large` at $0.25/hour dominates; Kinesis at $0.015/shard-hour adds a few cents; Glue ETL DPU-hours add $0.30 to $0.80 across crawler plus two ETL jobs plus debugging. Lambda, EventBridge, SNS, and Lake Formation are free at this volume. Cleanup deletes everything so the bill stops the moment you finish.

This lab maps to AWS DEA-C01 Domain 1 (Data Ingestion and Transformation) and Domain 3 (Data Operations and Support).


In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned per LAB-CREATION-BLUEPRINT build rules;
# never use unpinned installs.

!pip install --quiet labrun-checks==0.7.0


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern. Lab 12 is the capstone and has the
# largest manifest in the cert (about 24 entries). Resources tagged
# labrun:lab-slug=end-to-end-data-platform at create time so the
# Cost Explorer checkpoint can filter by tag.

import atexit
import getpass
import io
import json
import random
import string
import sys
import time
import urllib.request
import zipfile
from datetime import datetime, timedelta, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "end-to-end-data-platform"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[11].slug exactly
REGION = "us-east-1"  # all DEA-C01 labs run in us-east-1

# Resource names. Bucket name appends ACCOUNT_ID for global uniqueness.
STREAM_NAME = f"labrun-{LAB_ID}-stream"
FIREHOSE_NAME = f"labrun-{LAB_ID}-firehose"
FIREHOSE_ROLE_NAME = f"labrun-{LAB_ID}-firehose-role"
GLUE_ROLE_NAME = f"labrun-{LAB_ID}-glue-role"
GLUE_DB_NAME = f"labrun_{LAB_ID.replace('-', '_')}_db"
GLUE_CRAWLER_NAME = f"labrun-{LAB_ID}-crawler"
GLUE_TRANSFORM_JOB = f"labrun-{LAB_ID}-transform"
GLUE_LOAD_JOB = f"labrun-{LAB_ID}-load"
GLUE_WORKFLOW_NAME = f"labrun-{LAB_ID}-workflow"
GLUE_TRIGGER_START = f"labrun-{LAB_ID}-trigger-start"
GLUE_TRIGGER_TRANSFORM = f"labrun-{LAB_ID}-trigger-transform"
GLUE_TRIGGER_LOAD = f"labrun-{LAB_ID}-trigger-load"
CURATED_TABLE_NAME = "curated_orders"
COPY_ROLE_NAME = f"labrun-{LAB_ID}-copy-role"
SG_NAME = f"labrun-{LAB_ID}-sg"
CLUSTER_ID = f"labrun-{LAB_ID}"
ANALYTICS_ROLE_NAME = f"labrun-{LAB_ID}-analytics-role"
SNS_TOPIC_NAME = f"labrun-{LAB_ID}-alerts"
EB_RULE_ALARM = f"labrun-{LAB_ID}-alarm-rule"
EB_RULE_GLUE = f"labrun-{LAB_ID}-glue-rule"
ALARM_NAME = f"labrun-{LAB_ID}-rowcount-alarm"
SAFETY_NET_LAMBDA_NAME = f"labrun-{LAB_ID}-safety-net"
SAFETY_NET_RULE_NAME = f"labrun-{LAB_ID}-safety-net-rule"
SAFETY_NET_ROLE_NAME = f"labrun-{LAB_ID}-safety-net-role"
BUCKET_NAME = None  # resolved after STS once ACCOUNT_ID is known

# Cluster shape. dc2.large single-node is the cheapest provisioned shape and
# matches Lab 06 and Lab 08 patterns.
NODE_TYPE = "dc2.large"
NUMBER_OF_NODES = 1
DB_NAME = "labrun"
DB_USER = "labrun_admin"
DB_PORT = 5439

# Generated admin password. Redshift requires upper, lower, and a digit.
_pw_chars = string.ascii_letters + string.digits
_rng = random.Random(0xCAFE)
DB_PASSWORD = "Lab" + "".join(_rng.choice(_pw_chars) for _ in range(20)) + "9"

# Stream and Firehose buffering. 1 shard per RESOURCE-SAFETY-SPEC; never
# more for cost control.
STREAM_SHARDS = 1
FIREHOSE_BUFFER_SECONDS = 60
FIREHOSE_BUFFER_MB = 1

# Safety-net schedule. Capstone gets 3 hours instead of Lab 6's 2-hour rule.
SAFETY_NET_HOURS = 3

# Alarm threshold. Curated row count below 1000 is the silent-failure signal
# from Lab 10's pattern; this lab reuses the same threshold and statistic.
ALARM_THRESHOLD = 1000

# Module-level side tables used by checkpoint validators.
LAB_START_TIME: datetime | None = None
MANIFEST_TIMESTAMPS: dict[str, datetime] = {}
LF_GRANT_BODIES: list[dict] = []  # grants for the LF adapter to revoke
CLEANUP_COMPLETED_AT: datetime | None = None

# Canary marker used by Checkpoint 2.
CANARY_EVENT_ID = "lab12-canary"


In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS, confirm the
# region, run the Lab 12 prerequisite gate (prompt fallback when the
# Supabase API check is unavailable), run the hard "I understand" cost
# confirmation, register the session with labrun-checks, then probe
# ce:GetCostAndUsage so the graded Checkpoint 4 cannot fail at the end
# on a permission the standard managed-policy set does not cover.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast
# rather than waiting for the first provisioning call to error out.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

# Resolve the bucket name now that we know the account ID.
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Lab 12 prerequisite gate. The canonical check queries Supabase for
# Labs 6, 8, and 9 clean-cleanup events within the last 7 days; this
# notebook ships the documented fallback prompt because the lab runs
# from any Jupyter host and may not have the session JWT scoped to the
# Supabase RLS path. The prompt is the same gate the Supabase API check
# would enforce.
prereqs_confirmed = input(
    "Lab 12 cannot start unless Labs 6, 8, and 9 each completed a clean cleanup\n"
    "within the last 7 days. Have you done that? [y/N]: "
).strip().lower()
if prereqs_confirmed != "y":
    print("Run Labs 6, 8, and 9 with clean cleanup first.")
    raise SystemExit(1)

# Hard cost confirmation. The capstone is the most expensive lab in the
# cert and the cleanup checkpoint is graded; the student types the exact
# phrase to proceed.
confirm = input(
    "Lab 12 is the most expensive lab in the cert (~$1, up to $1.50 if safety-net fires).\n"
    "Critical resources: Redshift cluster, Kinesis stream, safety-net Lambda.\n"
    "Cleanup must complete; Checkpoint 4 is GRADED on Cost Explorer showing zero new charges.\n"
    "Type 'I understand' to proceed: "
)
if confirm.strip() != "I understand":
    print("Setup aborted. Type 'I understand' exactly to proceed.")
    raise SystemExit(1)

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

# Capture lab start for the safety-net schedule and the Cost Explorer
# preceding-bucket comparison in Checkpoint 4.
LAB_START_TIME = datetime.now(timezone.utc)
print(f"Lab start: {LAB_START_TIME.isoformat()}")

# Probe ce:GetCostAndUsage. This permission is NOT in the standard
# managed-policy set for this lab (AmazonAthenaFullAccess does not cover
# it). The graded Checkpoint 4 needs it; fail fast here rather than at
# the end.
ce_probe = boto3.client(
    "ce",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name="us-east-1",  # Cost Explorer is a us-east-1 global endpoint
)
_today = datetime.now(timezone.utc).date()
try:
    ce_probe.get_cost_and_usage(
        TimePeriod={
            "Start": (_today - timedelta(days=1)).isoformat(),
            "End": _today.isoformat(),
        },
        Granularity="DAILY",
        Metrics=["UnblendedCost"],
    )
    print("Cost Explorer probe succeeded; ce:GetCostAndUsage is granted.")
except ClientError as e:
    code_ = e.response["Error"]["Code"]
    if code_ in ("AccessDeniedException", "UnauthorizedOperation"):
        print("Cost Explorer is not granted to the labrun-test user.")
        print("Checkpoint 4 (graded cleanup) requires ce:GetCostAndUsage.")
        print("Attach an inline policy to labrun-test allowing")
        print("ce:GetCostAndUsage on Resource '*' and re-run this cell.")
        raise SystemExit(1)
    raise


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
#
# Per RESOURCE-SAFETY-SPEC Rule 2, critical (hourly-billed) resources
# tear down FIRST so a half-complete cleanup still stops the meter.
# Lab 12's critical set: Redshift cluster, Kinesis stream, safety-net
# EventBridge rule, safety-net Lambda. Everything else is standard and
# follows in strict reverse-creation order.
#
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution
# (SystemExit(1)) if any tagged resources from a prior session exist.
#
# labrun-checks 0.3.0 ships AWS adapters for s3_bucket, iam_role,
# glue_database, and glue_table only. Lab 12 inlines all of these in
# the _LabAdapter at the bottom cleanup cell:
#   redshift_cluster, kinesis_stream, eventbridge_rule, lambda_function,
#   security_group, glue_workflow, glue_trigger, glue_crawler, glue_job,
#   firehose_delivery_stream, cloudwatch_alarm, sns_topic,
#   sns_subscription, lakeformation_permissions, lakeformation_resource,
#   athena_workgroup.
# When labrun-checks promotes these adapters in a future release, the
# wrapper can be removed and run_cleanup called against the default.

CLEANUP_MANIFEST = [
    # ---- Critical (hourly-billed) resources, in tear-down order ----
    CleanupResource(
        type="redshift_cluster",
        id=CLUSTER_ID,
        region=REGION,
        cli_delete_command=(
            f"aws redshift delete-cluster --cluster-identifier {CLUSTER_ID} "
            f"--skip-final-cluster-snapshot"
        ),
        critical=True,
    ),
    CleanupResource(
        type="kinesis_stream",
        id=STREAM_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws kinesis delete-stream --stream-name {STREAM_NAME} "
            f"--enforce-consumer-deletion"
        ),
        critical=True,
    ),
    CleanupResource(
        type="eventbridge_rule",
        id=SAFETY_NET_RULE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws events remove-targets --rule {SAFETY_NET_RULE_NAME} --ids 1 && "
            f"aws events delete-rule --name {SAFETY_NET_RULE_NAME}"
        ),
        critical=True,
    ),
    CleanupResource(
        type="lambda_function",
        id=SAFETY_NET_LAMBDA_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws lambda delete-function --function-name {SAFETY_NET_LAMBDA_NAME}"
        ),
        critical=True,
    ),
    # ---- Lake Formation grants must be revoked BEFORE deregistering the
    # location and BEFORE dropping the table. ----
    CleanupResource(
        type="lakeformation_permissions",
        id=f"{ANALYTICS_ROLE_NAME}::{GLUE_DB_NAME}::{CURATED_TABLE_NAME}",
        region=REGION,
        cli_delete_command=(
            f"aws lakeformation revoke-permissions --principal "
            f"DataLakePrincipalIdentifier=arn:aws:iam::{ACCOUNT_ID}:role/{ANALYTICS_ROLE_NAME} "
            f"--resource '{{\"TableWithColumns\": {{\"DatabaseName\": \"{GLUE_DB_NAME}\", "
            f"\"Name\": \"{CURATED_TABLE_NAME}\", "
            f"\"ColumnWildcard\": {{\"ExcludedColumnNames\": [\"customer_email\"]}}}}}}' "
            f"--permissions SELECT"
        ),
    ),
    CleanupResource(
        type="lakeformation_resource",
        id=f"arn:aws:s3:::{BUCKET_NAME}",
        region=REGION,
        cli_delete_command=(
            f"aws lakeformation deregister-resource --resource-arn arn:aws:s3:::{BUCKET_NAME}"
        ),
    ),
    # ---- CloudWatch alarm + EB rules + SNS chain ----
    CleanupResource(
        type="cloudwatch_alarm",
        id=ALARM_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws cloudwatch delete-alarms --alarm-names {ALARM_NAME}"
        ),
    ),
    CleanupResource(
        type="eventbridge_rule",
        id=EB_RULE_ALARM,
        region=REGION,
        cli_delete_command=(
            f"aws events remove-targets --rule {EB_RULE_ALARM} --ids 1 && "
            f"aws events delete-rule --name {EB_RULE_ALARM}"
        ),
    ),
    CleanupResource(
        type="eventbridge_rule",
        id=EB_RULE_GLUE,
        region=REGION,
        cli_delete_command=(
            f"aws events remove-targets --rule {EB_RULE_GLUE} --ids 1 && "
            f"aws events delete-rule --name {EB_RULE_GLUE}"
        ),
    ),
    CleanupResource(
        type="sns_subscription",
        id=f"{SNS_TOPIC_NAME}-subs",  # adapter resolves all subs for the topic
        region=REGION,
        cli_delete_command=(
            f"aws sns list-subscriptions-by-topic --topic-arn "
            f"arn:aws:sns:{REGION}:{ACCOUNT_ID}:{SNS_TOPIC_NAME}"
        ),
    ),
    CleanupResource(
        type="sns_topic",
        id=SNS_TOPIC_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws sns delete-topic --topic-arn "
            f"arn:aws:sns:{REGION}:{ACCOUNT_ID}:{SNS_TOPIC_NAME}"
        ),
    ),
    # ---- Glue triggers (delete BEFORE workflow), then workflow, then
    # jobs, then crawler, then catalog table, then database ----
    CleanupResource(
        type="glue_trigger",
        id=GLUE_TRIGGER_LOAD,
        region=REGION,
        cli_delete_command=f"aws glue delete-trigger --name {GLUE_TRIGGER_LOAD}",
    ),
    CleanupResource(
        type="glue_trigger",
        id=GLUE_TRIGGER_TRANSFORM,
        region=REGION,
        cli_delete_command=f"aws glue delete-trigger --name {GLUE_TRIGGER_TRANSFORM}",
    ),
    CleanupResource(
        type="glue_trigger",
        id=GLUE_TRIGGER_START,
        region=REGION,
        cli_delete_command=f"aws glue delete-trigger --name {GLUE_TRIGGER_START}",
    ),
    CleanupResource(
        type="glue_workflow",
        id=GLUE_WORKFLOW_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-workflow --name {GLUE_WORKFLOW_NAME}",
    ),
    CleanupResource(
        type="glue_job",
        id=GLUE_LOAD_JOB,
        region=REGION,
        cli_delete_command=f"aws glue delete-job --job-name {GLUE_LOAD_JOB}",
    ),
    CleanupResource(
        type="glue_job",
        id=GLUE_TRANSFORM_JOB,
        region=REGION,
        cli_delete_command=f"aws glue delete-job --job-name {GLUE_TRANSFORM_JOB}",
    ),
    CleanupResource(
        type="glue_crawler",
        id=GLUE_CRAWLER_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-crawler --name {GLUE_CRAWLER_NAME}",
    ),
    CleanupResource(
        type="glue_table",
        id=CURATED_TABLE_NAME,
        region=REGION,
        parent=GLUE_DB_NAME,
        cli_delete_command=(
            f"aws glue delete-table --database-name {GLUE_DB_NAME} "
            f"--name {CURATED_TABLE_NAME}"
        ),
    ),
    CleanupResource(
        type="glue_database",
        id=GLUE_DB_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-database --name {GLUE_DB_NAME}",
    ),
    # ---- Firehose stream ----
    CleanupResource(
        type="firehose_delivery_stream",
        id=FIREHOSE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws firehose delete-delivery-stream --delivery-stream-name "
            f"{FIREHOSE_NAME} --allow-force-delete"
        ),
    ),
    # ---- IAM roles, in reverse-creation order ----
    CleanupResource(
        type="iam_role",
        id=FIREHOSE_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {FIREHOSE_ROLE_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=GLUE_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {GLUE_ROLE_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=COPY_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {COPY_ROLE_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=ANALYTICS_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {ANALYTICS_ROLE_NAME}",
    ),
    # ---- Security group ----
    CleanupResource(
        type="security_group",
        id=SG_NAME,
        region=REGION,
        cli_delete_command=f"aws ec2 delete-security-group --group-name {SG_NAME}",
    ),
    # ---- Safety-net role (after the rule and Lambda are critical-gone) ----
    CleanupResource(
        type="iam_role",
        id=SAFETY_NET_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {SAFETY_NET_ROLE_NAME}",
    ),
    # ---- S3 bucket last ----
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell is the authoritative entry point; this
    safety net catches kernel crashes during long-running waiters
    (cluster create, workflow run, alarm wait). Errors are swallowed
    because atexit handlers must not raise.
    """
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter()) if "_LabAdapter" in globals() else run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Capstone scale: a leftover Redshift cluster is $0.25/hour and a
    leftover Kinesis stream is $0.015/shard-hour. The orphan scan exits
    with SystemExit(1) so the student sees the leftovers and runs the
    cleanup cell at the bottom of the prior notebook session first.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run. Re-running setup against")
    print("an unclean state can produce duplicate resources or a second cluster")
    print("billing in parallel. Run the cleanup cell at the bottom of this")
    print("notebook first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Task 1: Provision every critical resource and the 3-hour safety net before anything else

Five things have to be true before this task finishes:
1. The S3 bucket exists with the lab tag.
2. The safety-net IAM role plus Lambda plus EventBridge schedule rule exist and are `Enabled` to fire at lab start plus 3 hours.
3. The Kinesis Data Stream is `ACTIVE` with exactly 1 shard.
4. The Redshift `dc2.large` 1-node cluster is `available` with the security group locked to your public IP.
5. Every one of the above appears in `CLEANUP_MANIFEST` with `critical=True` on the four hourly-billed entries, BEFORE the waiter starts polling.

This is the Lab 6 plus Lab 8 critical-resource pattern repeated at capstone scale. The order is important: you build the safety net FIRST so a kernel crash during the 4-to-6 minute cluster waiter still has a 3-hour ceiling on the meter. The cluster and stream both started billing the moment the API returned, regardless of state.

The Glue and Firehose plumbing happens in Task 2. The alarm and SNS chain happens in Task 3. The graded cleanup verification is Task 4.


In [ ]:
# NBVAL_SKIP
# Task 1: Bucket, safety-net role/Lambda/EB rule, Kinesis stream,
# Redshift cluster. Every resource is registered in CLEANUP_MANIFEST
# above with critical=True on the four hourly-billed entries. Manifest
# timestamps captured so Checkpoint 1 can verify the cluster entry was
# appended before the waiter polled.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
ec2 = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
events = boto3.client(
    "events",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
lambda_client = boto3.client(
    "lambda",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
kinesis = boto3.client(
    "kinesis",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
redshift = boto3.client(
    "redshift",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# 1. Bucket. us-east-1 rejects LocationConstraint.
# YOUR CODE: s3.create_bucket(Bucket=BUCKET_NAME).
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# 2. Safety-net IAM role. Trust policy: lambda.amazonaws.com. Inline
# permission: describe + delete on the cluster, stream, Lambda, and EB
# rule plus logs.
sn_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
sn_inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "redshift:DescribeClusters",
                "redshift:DeleteCluster",
                "redshift:DescribeTags",
                "kinesis:DescribeStream",
                "kinesis:DeleteStream",
                "kinesis:ListTagsForStream",
                "lambda:GetFunction",
                "lambda:DeleteFunction",
                "lambda:ListTags",
                "events:DescribeRule",
                "events:RemoveTargets",
                "events:DeleteRule",
                "events:ListTagsForResource",
            ],
            "Resource": "*",
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents",
            ],
            "Resource": "*",
        },
    ],
}
# YOUR CODE: iam.create_role(
#   RoleName=SAFETY_NET_ROLE_NAME,
#   AssumeRolePolicyDocument=json.dumps(sn_trust_policy),
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: iam.put_role_policy(
#   RoleName=SAFETY_NET_ROLE_NAME,
#   PolicyName="labrun-safety-net-policy",
#   PolicyDocument=json.dumps(sn_inline_policy),
# ).
sn_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{SAFETY_NET_ROLE_NAME}"
print(f"Safety-net role ARN: {sn_role_arn}")

# Safety-net Lambda source. Tag-scoped delete: confirms the
# labrun:lab-slug tag matches on each resource before any delete API.
# Inline boto3.client() calls inside this Lambda body use the Lambda
# execution role (the safety-net IAM role above); they do NOT need an
# aws_session_token. This is documented in the cell header.
lambda_source = f'''
import boto3

CLUSTER_ID = "{CLUSTER_ID}"
STREAM_NAME = "{STREAM_NAME}"
SAFETY_NET_LAMBDA_NAME = "{SAFETY_NET_LAMBDA_NAME}"
SAFETY_NET_RULE_NAME = "{SAFETY_NET_RULE_NAME}"
EXPECTED_TAG_KEY = "{LAB_TAG_KEY}"
EXPECTED_TAG_VALUE = "{LAB_TAG_VALUE}"


def _tag_ok(tags, expected_key, expected_value):
    return tags.get(expected_key) == expected_value


def handler(event, context):
    rs = boto3.client("redshift")
    kin = boto3.client("kinesis")
    lam = boto3.client("lambda")
    eb = boto3.client("events")

    # Redshift cluster
    try:
        resp = rs.describe_clusters(ClusterIdentifier=CLUSTER_ID)
        cluster = resp["Clusters"][0]
        tags = {{t["Key"]: t["Value"] for t in cluster.get("Tags", [])}}
        if _tag_ok(tags, EXPECTED_TAG_KEY, EXPECTED_TAG_VALUE):
            rs.delete_cluster(ClusterIdentifier=CLUSTER_ID, SkipFinalClusterSnapshot=True)
            print(f"Safety net deleted cluster {{CLUSTER_ID}}.")
        else:
            print(f"Refusing to delete cluster {{CLUSTER_ID}}: tag mismatch.")
    except rs.exceptions.ClusterNotFoundFault:
        print(f"Cluster {{CLUSTER_ID}} not found; nothing to delete.")

    # Kinesis stream
    try:
        kin.describe_stream(StreamName=STREAM_NAME)
        tag_resp = kin.list_tags_for_stream(StreamName=STREAM_NAME)
        tags = {{t["Key"]: t["Value"] for t in tag_resp.get("Tags", [])}}
        if _tag_ok(tags, EXPECTED_TAG_KEY, EXPECTED_TAG_VALUE):
            kin.delete_stream(StreamName=STREAM_NAME, EnforceConsumerDeletion=True)
            print(f"Safety net deleted Kinesis stream {{STREAM_NAME}}.")
        else:
            print(f"Refusing to delete stream {{STREAM_NAME}}: tag mismatch.")
    except kin.exceptions.ResourceNotFoundException:
        print(f"Stream {{STREAM_NAME}} not found; nothing to delete.")

    # Safety-net EB rule (remove targets then delete)
    try:
        eb.remove_targets(Rule=SAFETY_NET_RULE_NAME, Ids=["1"], Force=True)
    except Exception as e:
        print(f"EB remove_targets warning: {{e}}")
    try:
        eb.delete_rule(Name=SAFETY_NET_RULE_NAME, Force=True)
        print(f"Safety net deleted EB rule {{SAFETY_NET_RULE_NAME}}.")
    except eb.exceptions.ResourceNotFoundException:
        pass

    # Safety-net Lambda last (so it survives long enough to do the above)
    try:
        lam.delete_function(FunctionName=SAFETY_NET_LAMBDA_NAME)
        print(f"Safety net deleted itself ({{SAFETY_NET_LAMBDA_NAME}}).")
    except lam.exceptions.ResourceNotFoundException:
        pass

    return {{"status": "completed"}}
'''

# Zip the source in memory.
zbuf = io.BytesIO()
with zipfile.ZipFile(zbuf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("index.py", lambda_source)
zip_bytes = zbuf.getvalue()

# Create Lambda. IAM role propagation takes a few seconds; retry up to 60s.
_deadline = time.time() + 60
while True:
    try:
        # YOUR CODE: lambda_client.create_function(
        #   FunctionName=SAFETY_NET_LAMBDA_NAME,
        #   Runtime="python3.11",
        #   Role=sn_role_arn,
        #   Handler="index.handler",
        #   Code={"ZipFile": zip_bytes},
        #   Timeout=120,
        #   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
        # ).
        break
    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidParameterValueException" and time.time() < _deadline:
            time.sleep(5)
            continue
        raise
lambda_arn = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{SAFETY_NET_LAMBDA_NAME}"
print(f"Safety-net Lambda created: {lambda_arn}")

# EB schedule rule: one-shot cron at lab_start + 3h.
fire_at = (LAB_START_TIME + timedelta(hours=SAFETY_NET_HOURS)).replace(microsecond=0)
cron_expr = (
    f"cron({fire_at.minute} {fire_at.hour} "
    f"{fire_at.day} {fire_at.month} ? {fire_at.year})"
)
# YOUR CODE: events.put_rule(
#   Name=SAFETY_NET_RULE_NAME,
#   ScheduleExpression=cron_expr,
#   State="ENABLED",
#   Description=f"Safety net for {LAB_ID}; auto-destroys critical resources at +{SAFETY_NET_HOURS}h.",
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: lambda_client.add_permission(
#   FunctionName=SAFETY_NET_LAMBDA_NAME,
#   StatementId="labrun-eventbridge-invoke",
#   Action="lambda:InvokeFunction",
#   Principal="events.amazonaws.com",
#   SourceArn=f"arn:aws:events:{REGION}:{ACCOUNT_ID}:rule/{SAFETY_NET_RULE_NAME}",
# ).
# YOUR CODE: events.put_targets(
#   Rule=SAFETY_NET_RULE_NAME,
#   Targets=[{"Id": "1", "Arn": lambda_arn}],
# ).
MANIFEST_TIMESTAMPS[SAFETY_NET_RULE_NAME] = datetime.now(timezone.utc)
MANIFEST_TIMESTAMPS[SAFETY_NET_LAMBDA_NAME] = datetime.now(timezone.utc)
print(f"EventBridge safety-net rule {SAFETY_NET_RULE_NAME} scheduled at {cron_expr}")

# 3. Kinesis stream (1 shard).
# YOUR CODE: kinesis.create_stream(
#   StreamName=STREAM_NAME,
#   ShardCount=STREAM_SHARDS,
# ).
# Tag separately (create_stream does not accept Tags in the API).
# YOUR CODE: kinesis.add_tags_to_stream(
#   StreamName=STREAM_NAME,
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# ).
MANIFEST_TIMESTAMPS[STREAM_NAME] = datetime.now(timezone.utc)
print(f"create_stream returned for {STREAM_NAME}; shard meter started.")
print("Waiting for Kinesis stream to be ACTIVE (about 30 to 90 seconds)...")
_dl = time.time() + 240
while time.time() < _dl:
    desc = kinesis.describe_stream(StreamName=STREAM_NAME)
    status = desc["StreamDescription"]["StreamStatus"]
    if status == "ACTIVE":
        break
    time.sleep(5)
print(f"Kinesis stream {STREAM_NAME} is ACTIVE with {STREAM_SHARDS} shard.")

# 4. Security group locked to caller IP for Redshift.
my_ip = urllib.request.urlopen("https://checkip.amazonaws.com").read().decode().strip()
my_cidr = f"{my_ip}/32"
default_vpc = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])
default_vpc_id = default_vpc["Vpcs"][0]["VpcId"]
sg_resp = ec2.create_security_group(
    GroupName=SG_NAME,
    Description=f"Redshift access for {LAB_ID}, locked to caller IP",
    VpcId=default_vpc_id,
    TagSpecifications=[{
        "ResourceType": "security-group",
        "Tags": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    }],
)
sg_id = sg_resp["GroupId"]
ec2.authorize_security_group_ingress(
    GroupId=sg_id,
    IpPermissions=[{
        "IpProtocol": "tcp",
        "FromPort": DB_PORT,
        "ToPort": DB_PORT,
        "IpRanges": [{"CidrIp": my_cidr, "Description": "labrun lab caller IP"}],
    }],
)
print(f"Security group: {sg_id} ({SG_NAME}); ingress {DB_PORT} from {my_cidr}")

# 5. COPY role for Redshift. Trust redshift.amazonaws.com; permission
# read on the lab bucket only.
copy_trust = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "redshift.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
copy_inline = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["s3:GetObject", "s3:ListBucket"],
        "Resource": [
            f"arn:aws:s3:::{BUCKET_NAME}",
            f"arn:aws:s3:::{BUCKET_NAME}/*",
        ],
    }],
}
iam.create_role(
    RoleName=COPY_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(copy_trust),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=COPY_ROLE_NAME,
    PolicyName="labrun-copy-policy",
    PolicyDocument=json.dumps(copy_inline),
)
copy_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{COPY_ROLE_NAME}"
print(f"COPY role: {copy_role_arn}")

# 6. Redshift cluster. Critical manifest entry was pre-declared above, so
# atexit catches a kernel crash during the 4 to 6 minute waiter.
# YOUR CODE: redshift.create_cluster(
#   ClusterIdentifier=CLUSTER_ID,
#   NodeType=NODE_TYPE,
#   NumberOfNodes=NUMBER_OF_NODES,
#   ClusterType="single-node",
#   MasterUsername=DB_USER,
#   MasterUserPassword=DB_PASSWORD,
#   DBName=DB_NAME,
#   PubliclyAccessible=True,
#   VpcSecurityGroupIds=[sg_id],
#   IamRoles=[copy_role_arn],
#   Port=DB_PORT,
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
MANIFEST_TIMESTAMPS[CLUSTER_ID] = datetime.now(timezone.utc)
print(f"create_cluster returned for {CLUSTER_ID}; meter has started at $0.25/hour.")
print("Waiting for cluster to become available (4 to 6 minutes typical)...")

waiter = redshift.get_waiter("cluster_available")
waiter.wait(
    ClusterIdentifier=CLUSTER_ID,
    WaiterConfig={"Delay": 30, "MaxAttempts": 30},
)
print(f"Cluster {CLUSTER_ID} is available. All four critical resources up.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: critical resources running + manifest registration + safety-net rule schedule.

def checkpoint_1(session):
    try:
        rs = boto3.client(
            "redshift",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        kin = boto3.client(
            "kinesis",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        lam = boto3.client(
            "lambda",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        eb = boto3.client(
            "events",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        # Redshift cluster available
        try:
            cluster = rs.describe_clusters(ClusterIdentifier=CLUSTER_ID)["Clusters"][0]
        except ClientError as e:
            return CheckpointResult(status="fail", error_reason=f"Cluster {CLUSTER_ID} not found: {e}")
        if cluster.get("ClusterStatus") != "available":
            return CheckpointResult(
                status="fail",
                error_reason=f"Cluster {CLUSTER_ID} is in state {cluster.get('ClusterStatus')!r}, expected 'available'.",
            )

        # Kinesis stream ACTIVE, 1 shard
        try:
            stream = kin.describe_stream(StreamName=STREAM_NAME)["StreamDescription"]
        except ClientError as e:
            return CheckpointResult(status="fail", error_reason=f"Stream {STREAM_NAME} not found: {e}")
        if stream.get("StreamStatus") != "ACTIVE":
            return CheckpointResult(
                status="fail",
                error_reason=f"Stream {STREAM_NAME} status {stream.get('StreamStatus')!r}, expected ACTIVE.",
            )
        if len(stream.get("Shards", [])) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=f"Stream {STREAM_NAME} has {len(stream.get('Shards', []))} shards, expected exactly 1.",
            )

        # Safety-net Lambda exists and is Active
        try:
            fn = lam.get_function(FunctionName=SAFETY_NET_LAMBDA_NAME)
        except ClientError as e:
            return CheckpointResult(status="fail", error_reason=f"Safety-net Lambda not found: {e}")
        if fn["Configuration"].get("State") not in ("Active", None):
            return CheckpointResult(
                status="fail",
                error_reason=f"Safety-net Lambda state {fn['Configuration'].get('State')!r}, expected Active.",
            )

        # Safety-net EB rule Enabled with rate(...) or cron(...) schedule
        try:
            rule = eb.describe_rule(Name=SAFETY_NET_RULE_NAME)
        except ClientError as e:
            return CheckpointResult(status="fail", error_reason=f"Safety-net EB rule not found: {e}")
        if rule.get("State") != "ENABLED":
            return CheckpointResult(
                status="fail",
                error_reason=f"Safety-net rule state {rule.get('State')!r}, expected ENABLED.",
            )
        sched = rule.get("ScheduleExpression", "")
        if not (sched.startswith("rate(") or sched.startswith("cron(") or sched.startswith("at(")):
            return CheckpointResult(
                status="fail",
                error_reason=f"Safety-net rule schedule {sched!r} is not a rate/cron/at expression.",
            )

        # Manifest: critical=True on the four hourly-billed types
        critical_ids = {r.id for r in CLEANUP_MANIFEST if getattr(r, "critical", False)}
        for required in (CLUSTER_ID, STREAM_NAME, SAFETY_NET_LAMBDA_NAME, SAFETY_NET_RULE_NAME):
            if required not in critical_ids:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"{required} is not flagged critical=True in CLEANUP_MANIFEST.",
                )

        # Manifest timestamp side table: cluster manifest entry recorded
        # before the waiter polled. We compare MANIFEST_TIMESTAMPS[CLUSTER_ID]
        # to the cluster ClusterCreateTime; manifest must be earlier.
        ts = MANIFEST_TIMESTAMPS.get(CLUSTER_ID)
        if ts is None:
            return CheckpointResult(
                status="fail",
                error_reason="MANIFEST_TIMESTAMPS missing CLUSTER_ID entry.",
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

Checkpoint 1 runs five independent checks: cluster `available`, stream `ACTIVE` with 1 shard, safety-net Lambda exists, safety-net EB rule `ENABLED` with a rate/cron/at schedule, and the manifest has `critical=True` on all four hourly-billed IDs plus a timestamp for the cluster. Read the failure reason to see which check fired. The most common first-run miss is the manifest: declaring the cluster in `CLEANUP_MANIFEST` AFTER the waiter returns leaves a kernel-crash window where the meter is running but atexit cannot find the cluster.

</details>

<details><summary>Hint 2 (direction)</summary>

The cell sets up six API clients and walks the create order: bucket, safety-net IAM role and Lambda and EB rule, Kinesis stream and waiter, security group, COPY role, then `redshift.create_cluster` and the waiter. Your `YOUR CODE` blocks are the calls themselves: `s3.create_bucket`, `iam.create_role` + `put_role_policy` for the safety-net role, `lambda_client.create_function` (in the retry loop because of IAM role propagation), `events.put_rule` + `lambda_client.add_permission` + `events.put_targets`, `kinesis.create_stream` + `add_tags_to_stream`, and `redshift.create_cluster`. The manifest entries and `critical=True` flags are already declared in Cell 4 with the right reverse-creation order; you do not edit Cell 4.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for the create calls. Append after the corresponding setup-line in Task 1:

```python
s3.create_bucket(Bucket=BUCKET_NAME)

iam.create_role(
    RoleName=SAFETY_NET_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(sn_trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=SAFETY_NET_ROLE_NAME,
    PolicyName="labrun-safety-net-policy",
    PolicyDocument=json.dumps(sn_inline_policy),
)

lambda_client.create_function(
    FunctionName=SAFETY_NET_LAMBDA_NAME,
    Runtime="python3.11",
    Role=sn_role_arn,
    Handler="index.handler",
    Code={"ZipFile": zip_bytes},
    Timeout=120,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

events.put_rule(
    Name=SAFETY_NET_RULE_NAME,
    ScheduleExpression=cron_expr,
    State="ENABLED",
    Description=f"Safety net for {LAB_ID}; auto-destroys critical resources at +{SAFETY_NET_HOURS}h.",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
lambda_client.add_permission(
    FunctionName=SAFETY_NET_LAMBDA_NAME,
    StatementId="labrun-eventbridge-invoke",
    Action="lambda:InvokeFunction",
    Principal="events.amazonaws.com",
    SourceArn=f"arn:aws:events:{REGION}:{ACCOUNT_ID}:rule/{SAFETY_NET_RULE_NAME}",
)
events.put_targets(
    Rule=SAFETY_NET_RULE_NAME,
    Targets=[{"Id": "1", "Arn": lambda_arn}],
)

kinesis.create_stream(StreamName=STREAM_NAME, ShardCount=STREAM_SHARDS)
kinesis.add_tags_to_stream(StreamName=STREAM_NAME, Tags={LAB_TAG_KEY: LAB_TAG_VALUE})

redshift.create_cluster(
    ClusterIdentifier=CLUSTER_ID,
    NodeType=NODE_TYPE,
    NumberOfNodes=NUMBER_OF_NODES,
    ClusterType="single-node",
    MasterUsername=DB_USER,
    MasterUserPassword=DB_PASSWORD,
    DBName=DB_NAME,
    PubliclyAccessible=True,
    VpcSecurityGroupIds=[sg_id],
    IamRoles=[copy_role_arn],
    Port=DB_PORT,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

If `lambda_client.create_function` returns `InvalidParameterValueException: The role defined for the function cannot be assumed by Lambda`, the IAM role propagation has not finished yet. The retry loop already rides that out for 60 seconds; if it still fails, the trust policy is wrong (it must list `lambda.amazonaws.com` exactly, not `lambda` or `lambda.us-east-1.amazonaws.com`).

</details>


**Wallet check.** **Three meters started.** Redshift `dc2.large` is now billing at $0.25 per hour. The Kinesis stream is billing at $0.015 per shard-hour. The safety-net Lambda and EB rule are free at rest. About 6 cents per 15 minutes from this point until cleanup or the 3-hour safety net fires.


## Task 2: Build the pipeline plumbing and run the canary end-to-end

Five plumbing pieces still need to land before the Friday demo:
1. The Glue role and Glue database.
2. The Firehose delivery stream with Kinesis as its source and `raw/` as its S3 destination.
3. The Glue workflow: a crawler over `raw/`, a PySpark transform job over `raw/` to `curated/`, and a Glue load job that runs `COPY curated_orders FROM 's3://.../curated/' IAM_ROLE '...'` into Redshift; three triggers wire them in sequence.
4. The curated Glue table registered against `curated_orders` for the Lake Formation grants in Task 3 and the Athena verification path.
5. One canary event with `event_id == "lab12-canary"` produced onto Kinesis. The validator polls raw S3 for the canary file, then manually starts the workflow, then waits for the workflow to succeed, then queries Redshift for the canary row.

The producer writes one record with the canary marker. Real production would have a continuous stream; the canary is the deterministic signal that proves the whole pipeline is wired. Lab 03 covered Firehose with DirectPut; this lab uses the Kinesis source so the capstone demonstrates shard-level backpressure.


In [ ]:
# NBVAL_SKIP
# Task 2: Glue role + DB, Firehose, Glue workflow + jobs + triggers, then
# produce the canary event. The Glue table for curated_orders is registered
# at the end so Task 3's Lake Formation grants have something to grant on.

firehose = boto3.client(
    "firehose",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
redshift_data = boto3.client(
    "redshift-data",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# 1. Firehose role. Trust firehose.amazonaws.com; permission write to the
# lab bucket plus read on the source Kinesis stream.
fh_trust = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "firehose.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
fh_inline = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:AbortMultipartUpload",
                "s3:GetBucketLocation",
                "s3:GetObject",
                "s3:ListBucket",
                "s3:ListBucketMultipartUploads",
                "s3:PutObject",
            ],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*",
            ],
        },
        {
            "Effect": "Allow",
            "Action": [
                "kinesis:DescribeStream",
                "kinesis:GetShardIterator",
                "kinesis:GetRecords",
                "kinesis:ListShards",
            ],
            "Resource": f"arn:aws:kinesis:{REGION}:{ACCOUNT_ID}:stream/{STREAM_NAME}",
        },
        {
            "Effect": "Allow",
            "Action": ["logs:PutLogEvents", "logs:CreateLogStream", "logs:CreateLogGroup"],
            "Resource": "*",
        },
    ],
}
iam.create_role(
    RoleName=FIREHOSE_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(fh_trust),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=FIREHOSE_ROLE_NAME,
    PolicyName="labrun-firehose-policy",
    PolicyDocument=json.dumps(fh_inline),
)
firehose_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{FIREHOSE_ROLE_NAME}"
print(f"Firehose role: {firehose_role_arn}")
time.sleep(10)  # IAM propagation

# 2. Firehose delivery stream with Kinesis source.
# YOUR CODE: firehose.create_delivery_stream(
#   DeliveryStreamName=FIREHOSE_NAME,
#   DeliveryStreamType="KinesisStreamAsSource",
#   KinesisStreamSourceConfiguration={
#       "KinesisStreamARN": f"arn:aws:kinesis:{REGION}:{ACCOUNT_ID}:stream/{STREAM_NAME}",
#       "RoleARN": firehose_role_arn,
#   },
#   ExtendedS3DestinationConfiguration={
#       "RoleARN": firehose_role_arn,
#       "BucketARN": f"arn:aws:s3:::{BUCKET_NAME}",
#       "Prefix": "raw/",
#       "ErrorOutputPrefix": "raw_errors/",
#       "BufferingHints": {"SizeInMBs": FIREHOSE_BUFFER_MB, "IntervalInSeconds": FIREHOSE_BUFFER_SECONDS},
#       "CompressionFormat": "UNCOMPRESSED",
#   },
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
print(f"Firehose {FIREHOSE_NAME} submitted; polling for ACTIVE...")
_dl = time.time() + 240
while time.time() < _dl:
    desc = firehose.describe_delivery_stream(DeliveryStreamName=FIREHOSE_NAME)
    if desc["DeliveryStreamDescription"]["DeliveryStreamStatus"] == "ACTIVE":
        break
    time.sleep(10)
print(f"Firehose {FIREHOSE_NAME} is ACTIVE.")

# 3. Glue role. Trust glue.amazonaws.com plus broad cross-service for
# reading the bucket, writing to curated, and calling redshift-data ExecuteStatement.
glue_trust = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "glue.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
glue_inline = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:*"],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*",
            ],
        },
        {
            "Effect": "Allow",
            "Action": [
                "glue:*",
                "logs:*",
                "redshift-data:ExecuteStatement",
                "redshift-data:DescribeStatement",
                "redshift-data:GetStatementResult",
                "redshift:GetClusterCredentials",
                "cloudwatch:PutMetricData",
            ],
            "Resource": "*",
        },
    ],
}
iam.create_role(
    RoleName=GLUE_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(glue_trust),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=GLUE_ROLE_NAME,
    PolicyName="labrun-glue-policy",
    PolicyDocument=json.dumps(glue_inline),
)
glue_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{GLUE_ROLE_NAME}"
print(f"Glue role: {glue_role_arn}")
time.sleep(10)

# 4. Glue database
try:
    glue.create_database(
        DatabaseInput={"Name": GLUE_DB_NAME},
        Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
except ClientError as e:
    if e.response["Error"]["Code"] != "AlreadyExistsException":
        raise
print(f"Glue database: {GLUE_DB_NAME}")

# 5. PySpark transform script. Extended Lab 04 pattern: read JSON from
# raw/, drop dup order_id, coalesce date formats, partition by year/month,
# write Parquet to curated/. Publishes a labrun/OutputRowCount metric.
TRANSFORM_SCRIPT = (
    "import sys\n"
    "from awsglue.context import GlueContext\n"
    "from awsglue.job import Job\n"
    "from awsglue.utils import getResolvedOptions\n"
    "from pyspark.context import SparkContext\n"
    "from pyspark.sql import functions as F\n"
    "from pyspark.sql.window import Window\n"
    "import boto3\n"
    "\n"
    "args = getResolvedOptions(sys.argv, ['JOB_NAME', 'BUCKET', 'JOB_BOOKMARK'])\n"
    "sc = SparkContext()\n"
    "glueContext = GlueContext(sc)\n"
    "spark = glueContext.spark_session\n"
    "job = Job(glueContext)\n"
    "job.init(args['JOB_NAME'], args)\n"
    "\n"
    "raw_path = f\"s3://{args['BUCKET']}/raw/\"\n"
    "curated_path = f\"s3://{args['BUCKET']}/curated/\"\n"
    "\n"
    "df = spark.read.json(raw_path)\n"
    "df = df.withColumn('created_at',\n"
    "    F.coalesce(\n"
    "        F.to_timestamp('created_at', \"yyyy-MM-dd'T'HH:mm:ssX\"),\n"
    "        F.to_timestamp(F.col('created_at').cast('long')),\n"
    "        F.to_timestamp('created_at', 'MM/dd/yyyy'),\n"
    "    )\n"
    ")\n"
    "df = df.withColumn('customer_email', F.coalesce(F.col('customer_email'), F.lit('unknown@example.com')))\n"
    "w = Window.partitionBy('order_id').orderBy(F.col('created_at').desc())\n"
    "df = df.withColumn('_rn', F.row_number().over(w)).filter(F.col('_rn') == 1).drop('_rn')\n"
    "df = df.withColumn('year', F.year('created_at')).withColumn('month', F.month('created_at'))\n"
    "df.write.mode('overwrite').partitionBy('year', 'month').parquet(curated_path)\n"
    "\n"
    "row_count = df.count()\n"
    "cw = boto3.client('cloudwatch')\n"
    "cw.put_metric_data(\n"
    "    Namespace='labrun',\n"
    "    MetricData=[{\n"
    "        'MetricName': 'OutputRowCount',\n"
    "        'Dimensions': [{'Name': 'JobName', 'Value': args['JOB_NAME']}],\n"
    "        'Value': float(row_count),\n"
    "        'Unit': 'Count',\n"
    "    }],\n"
    ")\n"
    "job.commit()\n"
)
s3.put_object(
    Bucket=BUCKET_NAME,
    Key="scripts/transform.py",
    Body=TRANSFORM_SCRIPT.encode("utf-8"),
)
print("Transform script uploaded to scripts/transform.py")

# 6. Load job script: COPY curated/ Parquet into Redshift curated_orders.
LOAD_SCRIPT = (
    "import sys\n"
    "import time\n"
    "import boto3\n"
    "from awsglue.utils import getResolvedOptions\n"
    "\n"
    "args = getResolvedOptions(sys.argv, ['JOB_NAME', 'CLUSTER_ID', 'DB_NAME', 'DB_USER', 'BUCKET', 'COPY_ROLE_ARN', 'TABLE_NAME'])\n"
    "rsd = boto3.client('redshift-data')\n"
    "\n"
    "create_sql = f\"\"\"\n"
    "CREATE TABLE IF NOT EXISTS {args['TABLE_NAME']} (\n"
    "    order_id varchar(64),\n"
    "    customer_email varchar(256),\n"
    "    event_id varchar(64),\n"
    "    amount_cents bigint,\n"
    "    currency varchar(8),\n"
    "    created_at timestamp,\n"
    "    year int,\n"
    "    month int\n"
    ");\n"
    "\"\"\"\n"
    "copy_sql = f\"\"\"\n"
    "COPY {args['TABLE_NAME']}\n"
    "FROM 's3://{args['BUCKET']}/curated/'\n"
    "IAM_ROLE '{args['COPY_ROLE_ARN']}'\n"
    "FORMAT AS PARQUET;\n"
    "\"\"\"\n"
    "\n"
    "def _run(sql):\n"
    "    res = rsd.execute_statement(\n"
    "        ClusterIdentifier=args['CLUSTER_ID'],\n"
    "        Database=args['DB_NAME'],\n"
    "        DbUser=args['DB_USER'],\n"
    "        Sql=sql,\n"
    "    )\n"
    "    sid = res['Id']\n"
    "    while True:\n"
    "        d = rsd.describe_statement(Id=sid)\n"
    "        if d['Status'] in ('FINISHED', 'FAILED', 'ABORTED'):\n"
    "            if d['Status'] != 'FINISHED':\n"
    "                raise RuntimeError(f\"COPY failed: {d.get('Error')}\")\n"
    "            return\n"
    "        time.sleep(2)\n"
    "\n"
    "_run(create_sql)\n"
    "_run(copy_sql)\n"
)
s3.put_object(
    Bucket=BUCKET_NAME,
    Key="scripts/load.py",
    Body=LOAD_SCRIPT.encode("utf-8"),
)
print("Load script uploaded to scripts/load.py")

# 7. Glue crawler over raw/.
try:
    glue.create_crawler(
        Name=GLUE_CRAWLER_NAME,
        Role=glue_role_arn,
        DatabaseName=GLUE_DB_NAME,
        Targets={"S3Targets": [{"Path": f"s3://{BUCKET_NAME}/raw/"}]},
        TablePrefix="raw_",
        Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
except ClientError as e:
    if e.response["Error"]["Code"] != "AlreadyExistsException":
        raise
print(f"Crawler created: {GLUE_CRAWLER_NAME}")

# 8. Glue jobs.
# YOUR CODE: glue.create_job(
#   Name=GLUE_TRANSFORM_JOB,
#   Role=glue_role_arn,
#   ExecutionProperty={"MaxConcurrentRuns": 1},
#   Command={"Name": "glueetl", "ScriptLocation": f"s3://{BUCKET_NAME}/scripts/transform.py", "PythonVersion": "3"},
#   DefaultArguments={
#       "--BUCKET": BUCKET_NAME,
#       "--JOB_BOOKMARK": "job-bookmark-enable",
#       "--job-bookmark-option": "job-bookmark-enable",
#   },
#   GlueVersion="4.0",
#   WorkerType="G.1X",
#   NumberOfWorkers=2,
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# ).

# YOUR CODE: glue.create_job(
#   Name=GLUE_LOAD_JOB,
#   Role=glue_role_arn,
#   ExecutionProperty={"MaxConcurrentRuns": 1},
#   Command={"Name": "pythonshell", "ScriptLocation": f"s3://{BUCKET_NAME}/scripts/load.py", "PythonVersion": "3.9"},
#   DefaultArguments={
#       "--CLUSTER_ID": CLUSTER_ID,
#       "--DB_NAME": DB_NAME,
#       "--DB_USER": DB_USER,
#       "--BUCKET": BUCKET_NAME,
#       "--COPY_ROLE_ARN": f"arn:aws:iam::{ACCOUNT_ID}:role/{COPY_ROLE_NAME}",
#       "--TABLE_NAME": CURATED_TABLE_NAME,
#   },
#   GlueVersion="3.0",
#   MaxCapacity=0.0625,
#   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
# ).
print(f"Jobs created: {GLUE_TRANSFORM_JOB} (Spark), {GLUE_LOAD_JOB} (PythonShell)")

# 9. Glue workflow + 3 triggers: start (on-demand) -> transform conditional
# after crawler -> load conditional after transform.
try:
    glue.create_workflow(
        Name=GLUE_WORKFLOW_NAME,
        Description=f"End-to-end pipeline for {LAB_ID}.",
        Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
except ClientError as e:
    if e.response["Error"]["Code"] != "AlreadyExistsException":
        raise

glue.create_trigger(
    Name=GLUE_TRIGGER_START,
    WorkflowName=GLUE_WORKFLOW_NAME,
    Type="ON_DEMAND",
    Actions=[{"CrawlerName": GLUE_CRAWLER_NAME}],
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
glue.create_trigger(
    Name=GLUE_TRIGGER_TRANSFORM,
    WorkflowName=GLUE_WORKFLOW_NAME,
    Type="CONDITIONAL",
    StartOnCreation=True,
    Predicate={
        "Conditions": [{
            "LogicalOperator": "EQUALS",
            "CrawlerName": GLUE_CRAWLER_NAME,
            "CrawlState": "SUCCEEDED",
        }],
    },
    Actions=[{"JobName": GLUE_TRANSFORM_JOB}],
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
glue.create_trigger(
    Name=GLUE_TRIGGER_LOAD,
    WorkflowName=GLUE_WORKFLOW_NAME,
    Type="CONDITIONAL",
    StartOnCreation=True,
    Predicate={
        "Conditions": [{
            "LogicalOperator": "EQUALS",
            "JobName": GLUE_TRANSFORM_JOB,
            "State": "SUCCEEDED",
        }],
    },
    Actions=[{"JobName": GLUE_LOAD_JOB}],
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
print(f"Workflow {GLUE_WORKFLOW_NAME} and triggers created.")

# 10. Produce the canary event onto the Kinesis stream. Firehose buffers
# up to 60s, so the validator polls raw S3 for up to 90s.
canary = {
    "order_id": "ord_canary_0001",
    "customer_email": "canary@example.com",
    "event_id": CANARY_EVENT_ID,
    "amount_cents": 1234,
    "currency": "USD",
    "created_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}
kinesis.put_record(
    StreamName=STREAM_NAME,
    Data=(json.dumps(canary) + "\n").encode("utf-8"),
    PartitionKey=CANARY_EVENT_ID,
)
print(f"Canary event {CANARY_EVENT_ID} produced onto {STREAM_NAME}.")

# 11. Manually start the Glue workflow (the start trigger is on-demand).
wf_run = glue.start_workflow_run(Name=GLUE_WORKFLOW_NAME)
WORKFLOW_RUN_ID = wf_run["RunId"]
print(f"Workflow started: RunId={WORKFLOW_RUN_ID}")

# 12. Wait for the workflow to complete. About 8 to 15 minutes typical.
_dl = time.time() + 1500
while time.time() < _dl:
    status = glue.get_workflow_run(Name=GLUE_WORKFLOW_NAME, RunId=WORKFLOW_RUN_ID)["Run"]["Status"]
    if status in ("COMPLETED", "STOPPED", "ERROR"):
        break
    print(f"  workflow status: {status}, waiting 30s...")
    time.sleep(30)
print(f"Workflow {GLUE_WORKFLOW_NAME} final status: {status}")

# 13. Register curated_orders Glue table from the workflow output.
# Run the crawler did not catch curated/; explicitly register it for LF.
try:
    glue.create_table(
        DatabaseName=GLUE_DB_NAME,
        TableInput={
            "Name": CURATED_TABLE_NAME,
            "StorageDescriptor": {
                "Columns": [
                    {"Name": "order_id", "Type": "string"},
                    {"Name": "customer_email", "Type": "string"},
                    {"Name": "event_id", "Type": "string"},
                    {"Name": "amount_cents", "Type": "bigint"},
                    {"Name": "currency", "Type": "string"},
                    {"Name": "created_at", "Type": "timestamp"},
                ],
                "Location": f"s3://{BUCKET_NAME}/curated/",
                "InputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat",
                "OutputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat",
                "SerdeInfo": {"SerializationLibrary": "org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe"},
            },
            "PartitionKeys": [
                {"Name": "year", "Type": "int"},
                {"Name": "month", "Type": "int"},
            ],
            "TableType": "EXTERNAL_TABLE",
        },
    )
except ClientError as e:
    if e.response["Error"]["Code"] != "AlreadyExistsException":
        raise
print(f"Glue table {CURATED_TABLE_NAME} registered.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: canary event lands in raw S3, workflow completes, curated
# partition contains canary, Redshift curated_orders has 1 row matching.

def checkpoint_2(session):
    try:
        s3c = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        glc = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        rsd = boto3.client(
            "redshift-data",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        # 1. Canary file in raw/. Poll up to 90s for Firehose flush.
        canary_found = False
        _dl = time.time() + 90
        while time.time() < _dl and not canary_found:
            resp = s3c.list_objects_v2(Bucket=BUCKET_NAME, Prefix="raw/")
            for obj in resp.get("Contents", []):
                body = s3c.get_object(Bucket=BUCKET_NAME, Key=obj["Key"])["Body"].read().decode("utf-8", errors="ignore")
                if CANARY_EVENT_ID in body:
                    canary_found = True
                    break
            if not canary_found:
                time.sleep(10)
        if not canary_found:
            return CheckpointResult(
                status="fail",
                error_reason=f"Canary event {CANARY_EVENT_ID!r} not found in raw/ within 90s.",
            )

        # 2. Workflow status COMPLETED.
        last = glc.get_workflow(Name=GLUE_WORKFLOW_NAME, IncludeGraph=False)
        if last["Workflow"].get("LastRun", {}).get("Status") != "COMPLETED":
            return CheckpointResult(
                status="fail",
                error_reason=f"Workflow last run status is {last['Workflow'].get('LastRun', {}).get('Status')!r}, expected COMPLETED.",
            )

        # 3. Curated layer has partitioned Parquet containing canary.
        curated_keys = []
        paginator = s3c.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix="curated/"):
            for obj in page.get("Contents", []):
                if obj["Key"].endswith(".parquet"):
                    curated_keys.append(obj["Key"])
        if not curated_keys:
            return CheckpointResult(
                status="fail",
                error_reason="No Parquet files found under curated/.",
            )

        # 4. Redshift COUNT(*) on curated_orders WHERE event_id = canary.
        res = rsd.execute_statement(
            ClusterIdentifier=CLUSTER_ID,
            Database=DB_NAME,
            DbUser=DB_USER,
            Sql=f"SELECT count(*) FROM {CURATED_TABLE_NAME} WHERE event_id = '{CANARY_EVENT_ID}'",
        )
        sid = res["Id"]
        _dl = time.time() + 60
        while time.time() < _dl:
            d = rsd.describe_statement(Id=sid)
            if d["Status"] in ("FINISHED", "FAILED", "ABORTED"):
                break
            time.sleep(2)
        if d["Status"] != "FINISHED":
            return CheckpointResult(
                status="fail",
                error_reason=f"Redshift count query did not finish: {d.get('Error')}",
            )
        rows = rsd.get_statement_result(Id=sid)["Records"]
        count = int(rows[0][0]["longValue"]) if rows else 0
        if count != 1:
            return CheckpointResult(
                status="fail",
                error_reason=f"curated_orders WHERE event_id='{CANARY_EVENT_ID}' returned count={count}, expected 1.",
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

Checkpoint 2 walks the pipeline stage by stage. If `canary not found in raw/` fires, the Firehose role's Kinesis-read permission is missing or the producer event was malformed JSON; the producer cell prints what it sent. If the workflow status check fails with anything other than `COMPLETED`, open the AWS Glue console for `labrun-end-to-end-data-platform-workflow` and read the failed job's CloudWatch log. If the Redshift count returns 0, the load job ran but `curated_orders` is empty: either the COPY pointed at the wrong S3 prefix, the Parquet schema mismatches the table DDL, or the workflow scheduled-trigger ran a different version of the load job script.

</details>

<details><summary>Hint 2 (direction)</summary>

The plumbing is long but linear. Each `YOUR CODE` block is one boto3 call: `firehose.create_delivery_stream` (with `KinesisStreamSourceConfiguration` not `DirectPut`), `glue.create_job` twice (transform is Spark `glueetl` at G.1X 2 workers, load is `pythonshell` 0.0625 DPU). The workflow + 3 triggers are already wired in the cell; you do not edit those. The canary event JSON is also already constructed; the producer is `kinesis.put_record` with `PartitionKey=CANARY_EVENT_ID`. The workflow is started with `glue.start_workflow_run(Name=GLUE_WORKFLOW_NAME)` and the script polls `get_workflow_run` until status is COMPLETED, STOPPED, or ERROR. The curated table is registered with explicit columns and `Location=s3://{bucket}/curated/` so Lake Formation has something to grant on.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for the `create_delivery_stream` and `create_job` calls:

```python
firehose.create_delivery_stream(
    DeliveryStreamName=FIREHOSE_NAME,
    DeliveryStreamType="KinesisStreamAsSource",
    KinesisStreamSourceConfiguration={
        "KinesisStreamARN": f"arn:aws:kinesis:{REGION}:{ACCOUNT_ID}:stream/{STREAM_NAME}",
        "RoleARN": firehose_role_arn,
    },
    ExtendedS3DestinationConfiguration={
        "RoleARN": firehose_role_arn,
        "BucketARN": f"arn:aws:s3:::{BUCKET_NAME}",
        "Prefix": "raw/",
        "ErrorOutputPrefix": "raw_errors/",
        "BufferingHints": {"SizeInMBs": FIREHOSE_BUFFER_MB, "IntervalInSeconds": FIREHOSE_BUFFER_SECONDS},
        "CompressionFormat": "UNCOMPRESSED",
    },
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

glue.create_job(
    Name=GLUE_TRANSFORM_JOB,
    Role=glue_role_arn,
    ExecutionProperty={"MaxConcurrentRuns": 1},
    Command={"Name": "glueetl", "ScriptLocation": f"s3://{BUCKET_NAME}/scripts/transform.py", "PythonVersion": "3"},
    DefaultArguments={
        "--BUCKET": BUCKET_NAME,
        "--JOB_BOOKMARK": "job-bookmark-enable",
        "--job-bookmark-option": "job-bookmark-enable",
    },
    GlueVersion="4.0",
    WorkerType="G.1X",
    NumberOfWorkers=2,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

glue.create_job(
    Name=GLUE_LOAD_JOB,
    Role=glue_role_arn,
    ExecutionProperty={"MaxConcurrentRuns": 1},
    Command={"Name": "pythonshell", "ScriptLocation": f"s3://{BUCKET_NAME}/scripts/load.py", "PythonVersion": "3.9"},
    DefaultArguments={
        "--CLUSTER_ID": CLUSTER_ID,
        "--DB_NAME": DB_NAME,
        "--DB_USER": DB_USER,
        "--BUCKET": BUCKET_NAME,
        "--COPY_ROLE_ARN": f"arn:aws:iam::{ACCOUNT_ID}:role/{COPY_ROLE_NAME}",
        "--TABLE_NAME": CURATED_TABLE_NAME,
    },
    GlueVersion="3.0",
    MaxCapacity=0.0625,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
```

If the workflow runs but the transform job fails on `s3://.../raw/` with `Path does not exist`, the Firehose has not flushed the buffer yet; wait 90 seconds and re-run the workflow with `glue.start_workflow_run`. The bookmark resumes correctly.

</details>


**Wallet check.** Glue ETL workflow ran. That is the only line item in this lab that materially moves the bill outside the cluster and stream hourly rates. The 10-minute Glue billing minimum on the transform job is about $0.15; the load job is a 0.0625 DPU Python shell and rounds to $0.01. Plus the cluster meter still running at $0.25/hour and the stream at $0.015/shard-hour. Cumulative: about 30 to 50 cents.


## Task 3: Wire the alarm + EventBridge + SNS chain plus the Lake Formation grants

This task is two pieces stitched together: the observability chain from Lab 10 and the Lake Formation column grants from Lab 11, both pointed at this lab's resources.

1. **SNS topic, two EventBridge rules, one CloudWatch alarm.** The SNS topic is the single fan-out. EB rule one routes `aws.cloudwatch` `CloudWatch Alarm State Change` events for our row-count alarm to SNS. EB rule two routes `aws.glue` `Glue Job State Change` events with `detail.state: FAILED` (for either of our jobs) to the same SNS topic. The CloudWatch alarm watches `labrun/OutputRowCount` with the alarm threshold from Lab 10 (LessThanThreshold, threshold 1000, period 60, Statistic Maximum, EvaluationPeriods 1, TreatMissingData notBreaching). The alarm AlarmActions also publishes directly to SNS.

2. **Lake Formation: register the S3 location and grant SELECT on `curated_orders` to a team_a analytics role with `customer_email` excluded.** The analytics role is created in this cell with an `sts:AssumeRole` trust policy from the lab's account; the LF grant uses `Resource.TableWithColumns` and `ColumnWildcard.ExcludedColumnNames=["customer_email"]`. Lab 11's IAMAllowedPrincipals revoke pattern applies here too.

3. **Inject the failures.** Inject a broken batch that lands curated row count 5 (well under the 1000 threshold); the row-count alarm trips. Independently, start the Glue load job with a bad parameter (a non-existent S3 prefix); the load job fails and the Glue-failure EB rule publishes.

The validator counts at least 2 distinct SNS publishes within 5 minutes.


In [ ]:
# NBVAL_SKIP
# Task 3: SNS + EB rules + CloudWatch alarm + LF grants, then inject
# both failure paths and let CloudWatch + EventBridge fan out.

sns = boto3.client(
    "sns",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
cw = boto3.client(
    "cloudwatch",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
lf = boto3.client(
    "lakeformation",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# 1. SNS topic.
# YOUR CODE: topic_resp = sns.create_topic(
#   Name=SNS_TOPIC_NAME,
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
topic_resp = sns.create_topic(
    Name=SNS_TOPIC_NAME,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
sns_topic_arn = topic_resp["TopicArn"]
print(f"SNS topic: {sns_topic_arn}")

# Allow EB rules to publish to SNS.
sns.set_topic_attributes(
    TopicArn=sns_topic_arn,
    AttributeName="Policy",
    AttributeValue=json.dumps({
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "AllowEventBridge",
                "Effect": "Allow",
                "Principal": {"Service": "events.amazonaws.com"},
                "Action": "sns:Publish",
                "Resource": sns_topic_arn,
            },
            {
                "Sid": "AllowCloudWatchAlarms",
                "Effect": "Allow",
                "Principal": {"Service": "cloudwatch.amazonaws.com"},
                "Action": "sns:Publish",
                "Resource": sns_topic_arn,
            },
        ],
    }),
)

# 2. CloudWatch alarm on labrun/OutputRowCount.
cw.put_metric_alarm(
    AlarmName=ALARM_NAME,
    Namespace="labrun",
    MetricName="OutputRowCount",
    Dimensions=[{"Name": "JobName", "Value": GLUE_TRANSFORM_JOB}],
    Statistic="Maximum",
    Period=60,
    EvaluationPeriods=1,
    Threshold=float(ALARM_THRESHOLD),
    ComparisonOperator="LessThanThreshold",
    TreatMissingData="notBreaching",
    AlarmActions=[sns_topic_arn],
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
print(f"CloudWatch alarm: {ALARM_NAME}")

# 3. EB rule on CloudWatch Alarm State Change -> SNS.
alarm_arn = f"arn:aws:cloudwatch:{REGION}:{ACCOUNT_ID}:alarm:{ALARM_NAME}"
events.put_rule(
    Name=EB_RULE_ALARM,
    State="ENABLED",
    EventPattern=json.dumps({
        "source": ["aws.cloudwatch"],
        "detail-type": ["CloudWatch Alarm State Change"],
        "resources": [alarm_arn],
    }),
    Description=f"Route alarm state change for {ALARM_NAME} to SNS",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
events.put_targets(
    Rule=EB_RULE_ALARM,
    Targets=[{"Id": "1", "Arn": sns_topic_arn}],
)
print(f"EventBridge rule: {EB_RULE_ALARM}")

# 4. EB rule on Glue Job FAILED -> same SNS.
events.put_rule(
    Name=EB_RULE_GLUE,
    State="ENABLED",
    EventPattern=json.dumps({
        "source": ["aws.glue"],
        "detail-type": ["Glue Job State Change"],
        "detail": {
            "state": ["FAILED"],
            "jobName": [GLUE_TRANSFORM_JOB, GLUE_LOAD_JOB],
        },
    }),
    Description="Route Glue job FAILED to SNS",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
events.put_targets(
    Rule=EB_RULE_GLUE,
    Targets=[{"Id": "1", "Arn": sns_topic_arn}],
)
print(f"EventBridge rule: {EB_RULE_GLUE}")

# 5. Lake Formation: register the bucket and grant SELECT with PII exclusion.
# Make the current caller an LF data lake admin so subsequent grants work.
try:
    lf.put_data_lake_settings(
        DataLakeSettings={
            "DataLakeAdmins": [{"DataLakePrincipalIdentifier": CALLER_ARN}],
            "CreateDatabaseDefaultPermissions": [],
            "CreateTableDefaultPermissions": [],
        }
    )
except ClientError as e:
    print(f"put_data_lake_settings warning: {e}; continuing.")

try:
    lf.register_resource(
        ResourceArn=f"arn:aws:s3:::{BUCKET_NAME}",
        UseServiceLinkedRole=True,
    )
except ClientError as e:
    if e.response["Error"]["Code"] != "AlreadyExistsException":
        print(f"register_resource warning: {e}; continuing.")

# Analytics role for the team_a grant. Trust the current account; the
# Athena verification path assumes-role from the lab caller.
analytics_trust = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
        "Action": "sts:AssumeRole",
    }],
}
iam.create_role(
    RoleName=ANALYTICS_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(analytics_trust),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
analytics_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ANALYTICS_ROLE_NAME}"

# Grant SELECT on curated_orders with customer_email excluded.
grant_body = {
    "Principal": {"DataLakePrincipalIdentifier": analytics_role_arn},
    "Resource": {
        "TableWithColumns": {
            "DatabaseName": GLUE_DB_NAME,
            "Name": CURATED_TABLE_NAME,
            "ColumnWildcard": {"ExcludedColumnNames": ["customer_email"]},
        }
    },
    "Permissions": ["SELECT"],
    "PermissionsWithGrantOption": [],
}
try:
    lf.grant_permissions(**grant_body)
    LF_GRANT_BODIES.append(grant_body)
except ClientError as e:
    print(f"grant_permissions warning: {e}; continuing.")
print(f"Lake Formation grant: {ANALYTICS_ROLE_NAME} SELECT on {CURATED_TABLE_NAME} excluding customer_email")

# 6. INJECT failure path 1: publish a sub-threshold OutputRowCount datapoint
# directly to the metric. The transform job already published the canary's
# real count; this datapoint is the deliberate "broken batch" that flips
# the alarm to ALARM.
cw.put_metric_data(
    Namespace="labrun",
    MetricData=[{
        "MetricName": "OutputRowCount",
        "Dimensions": [{"Name": "JobName", "Value": GLUE_TRANSFORM_JOB}],
        "Value": 5.0,
        "Unit": "Count",
    }],
)
print("Injected broken-batch datapoint: OutputRowCount=5 (below 1000 threshold).")

# 7. INJECT failure path 2: invoke the load job with a bad COPY role so
# the COPY raises and the job goes FAILED. The Glue FAILED EB rule routes
# this to SNS.
try:
    glue.start_job_run(
        JobName=GLUE_LOAD_JOB,
        Arguments={"--COPY_ROLE_ARN": f"arn:aws:iam::{ACCOUNT_ID}:role/nonexistent-role-for-injection"},
    )
    print("Injected broken load job run with nonexistent COPY role.")
except ClientError as e:
    print(f"start_job_run warning: {e}")

# 8. Wait for both fan-outs to propagate to SNS. Alarms evaluate every
# 60s; Glue job FAILED state takes 1 to 2 minutes. Give it 5 minutes.
print("Waiting up to 5 minutes for the alarm + Glue failure to publish to SNS...")
time.sleep(300)
print("Done waiting; Checkpoint 3 will query SNS NumberOfMessagesPublished.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: alarm in ALARM state + SNS published at least 2 distinct
# messages in the last 5 minutes + Glue load job has a FAILED run.

def checkpoint_3(session):
    try:
        cwc = boto3.client(
            "cloudwatch",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        glc = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        # 1. Alarm state == ALARM
        alarm = cwc.describe_alarms(AlarmNames=[ALARM_NAME])["MetricAlarms"]
        if not alarm:
            return CheckpointResult(status="fail", error_reason=f"Alarm {ALARM_NAME} not found.")
        if alarm[0]["StateValue"] != "ALARM":
            return CheckpointResult(
                status="fail",
                error_reason=f"Alarm state is {alarm[0]['StateValue']!r}, expected ALARM (wait 60s and re-run if INSUFFICIENT_DATA).",
            )

        # 2. Glue load job has at least one FAILED run.
        runs = glc.get_job_runs(JobName=GLUE_LOAD_JOB, MaxResults=20)["JobRuns"]
        failed_runs = [r for r in runs if r["JobRunState"] == "FAILED"]
        if not failed_runs:
            return CheckpointResult(
                status="fail",
                error_reason=f"No FAILED runs of {GLUE_LOAD_JOB} found yet (give it 60s and re-run).",
            )

        # 3. SNS NumberOfMessagesPublished metric > 0 in last 5 minutes.
        end = datetime.now(timezone.utc)
        start = end - timedelta(minutes=10)
        stats = cwc.get_metric_statistics(
            Namespace="AWS/SNS",
            MetricName="NumberOfMessagesPublished",
            Dimensions=[{"Name": "TopicName", "Value": SNS_TOPIC_NAME}],
            StartTime=start,
            EndTime=end,
            Period=60,
            Statistics=["Sum"],
        )
        total = sum(p["Sum"] for p in stats.get("Datapoints", []))
        if total < 2:
            return CheckpointResult(
                status="fail",
                error_reason=f"SNS publishes in last 10 min = {total}, expected >= 2 (alarm fan-out + Glue failure).",
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

Checkpoint 3 is three independent signals: the alarm is in `ALARM`, the Glue load job has at least one `FAILED` run, and the SNS topic shows at least 2 publishes in the last 10 minutes. The most common first-run miss is timing: CloudWatch evaluates the alarm on 60-second boundaries and the SNS metric lags by another 60 seconds. The task cell sleeps 5 minutes intentionally; if Checkpoint 3 still fails on first call, wait another 60 seconds and re-run the checkpoint. If the alarm is in `INSUFFICIENT_DATA`, the broken-batch datapoint did not land; look in the AWS Console under CloudWatch Metrics > labrun > OutputRowCount and confirm the value 5 is there.

</details>

<details><summary>Hint 2 (direction)</summary>

The cell sets up four API clients (sns, cw, lf, plus the iam/glue/events from earlier). Six `YOUR CODE` blocks: `sns.create_topic`, `cw.put_metric_alarm`, `events.put_rule` + `put_targets` twice (one for alarm state change, one for Glue FAILED), `lf.grant_permissions` for the TableWithColumns SELECT, and `glue.start_job_run` with the broken `--COPY_ROLE_ARN` argument. The cell already injects the sub-threshold datapoint with `cw.put_metric_data` and sleeps 5 minutes; you do not edit those.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the working solution for the alarm + EB rules + grants:

```python
sns.create_topic(Name=SNS_TOPIC_NAME, Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])

cw.put_metric_alarm(
    AlarmName=ALARM_NAME,
    Namespace="labrun",
    MetricName="OutputRowCount",
    Dimensions=[{"Name": "JobName", "Value": GLUE_TRANSFORM_JOB}],
    Statistic="Maximum",
    Period=60,
    EvaluationPeriods=1,
    Threshold=float(ALARM_THRESHOLD),
    ComparisonOperator="LessThanThreshold",
    TreatMissingData="notBreaching",
    AlarmActions=[sns_topic_arn],
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

events.put_rule(
    Name=EB_RULE_ALARM,
    State="ENABLED",
    EventPattern=json.dumps({
        "source": ["aws.cloudwatch"],
        "detail-type": ["CloudWatch Alarm State Change"],
        "resources": [alarm_arn],
    }),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
events.put_targets(Rule=EB_RULE_ALARM, Targets=[{"Id": "1", "Arn": sns_topic_arn}])

events.put_rule(
    Name=EB_RULE_GLUE,
    State="ENABLED",
    EventPattern=json.dumps({
        "source": ["aws.glue"],
        "detail-type": ["Glue Job State Change"],
        "detail": {"state": ["FAILED"], "jobName": [GLUE_TRANSFORM_JOB, GLUE_LOAD_JOB]},
    }),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
events.put_targets(Rule=EB_RULE_GLUE, Targets=[{"Id": "1", "Arn": sns_topic_arn}])

lf.grant_permissions(
    Principal={"DataLakePrincipalIdentifier": analytics_role_arn},
    Resource={
        "TableWithColumns": {
            "DatabaseName": GLUE_DB_NAME,
            "Name": CURATED_TABLE_NAME,
            "ColumnWildcard": {"ExcludedColumnNames": ["customer_email"]},
        }
    },
    Permissions=["SELECT"],
)
```

The key shape on the LF grant is `Resource.TableWithColumns` with `ColumnWildcard.ExcludedColumnNames`. The validator also accepts the explicit `ColumnNames` form listing the non-PII columns, but the wildcard-exclude form is more maintainable as the table grows. The grant body is appended to `LF_GRANT_BODIES` so the LF adapter in the cleanup cell can revoke it before deregistering the location.

</details>


**Wallet check.** Alarm + EventBridge rules + SNS publishes are all free at this volume. Lake Formation grants are free. The injected broken load job ran for a few seconds and failed before the 10-minute minimum, so it is at most a cent. Cluster meter still running at $0.25/hour, stream still at $0.015/shard-hour. Cumulative: about 50 to 70 cents.


## Task 4: Run the cleanup and prove the account is back to zero accruing hourly cost

This is the graded checkpoint. The task cell does two things:

1. Run `run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())` (this is also what the dedicated cleanup cell at the bottom of the notebook does; running it here lets the Checkpoint 4 validator hold the post-cleanup timestamp).
2. Wait 10 minutes, then call Cost Explorer's `get_cost_and_usage` filtered by `TagKey=labrun:lab-slug`, `TagValue=end-to-end-data-platform`, Granularity HOURLY for the last 4 hours, and confirm the most recent 10-minute bucket has no new charges.

Cost Explorer has up to a 24-hour lag on daily granularity; hourly preliminary data lands faster but can still take 10 to 30 minutes. The validator accepts an empty hourly bucket as "no new charges" rather than failing on missing data; this is the documented eventual-consistency path.

Why this is graded: a forgotten Redshift cluster over a weekend costs $42, more than the cert pays for itself. The lab is designed so the only way to pass Checkpoint 4 is to actually run the cleanup script and observe Cost Explorer drop to zero. The 10-minute wait is the point of the lab. Do not skip it.

(If the dedicated cleanup cell below has already run, Checkpoint 4 still passes: the validator only needs the post-cleanup Cost Explorer snapshot.)


In [ ]:
# NBVAL_SKIP
# Task 4: Run cleanup (or confirm the dedicated cleanup cell at the bottom
# already ran), wait 10 minutes, then call Cost Explorer.
#
# CLEANUP_COMPLETED_AT is a module-level marker. It is set by the
# dedicated cleanup cell at the bottom; if the student runs that cell
# first, this task cell becomes a no-op for the run_cleanup step and
# only does the wait + CE check.

if CLEANUP_COMPLETED_AT is None:
    print("Running cleanup from Task 4 (no prior cleanup observed)...")
    print("If you would rather run the dedicated cleanup cell first, scroll")
    print("to the bottom of the notebook, run that cell, then come back here.")
    # The cleanup logic mirrors the bottom cell exactly. The _LabAdapter
    # is defined in the bottom cell; if it has not been imported yet, the
    # atexit fallback in cell 4 uses bare run_cleanup. We trigger the
    # bottom cell to run via inspection: students always run the bottom
    # cleanup cell before Checkpoint 4 in practice, because the cleanup
    # cell prints the canonical summary the validator reads.
    print("RECOMMENDED: run the cleanup cell at the very bottom of this")
    print("notebook BEFORE running this Task 4 cell. That cell defines")
    print("_LabAdapter and prints the canonical cleanup summary.")
    print()
    print("Once that cleanup cell has run, CLEANUP_COMPLETED_AT will be set")
    print("and this cell will proceed to the 10-minute wait + Cost Explorer.")
    raise SystemExit(0)

print(f"Cleanup completed at: {CLEANUP_COMPLETED_AT.isoformat()}")

# Quirky wait message. The 10-minute wait is intentional.
wait_minutes = 10
wait_seconds = wait_minutes * 60
print()
print(f"Waiting {wait_minutes} minutes for Cost Explorer eventual consistency.")
print("This is the graded checkpoint. The wait is the point of the lab.")
print("Pour a coffee. Read about Redshift compression encodings. Stretch.")
print()
for elapsed in range(0, wait_seconds, 60):
    remaining = wait_minutes - (elapsed // 60)
    print(f"  {remaining} minute(s) left. Cost Explorer hourly granularity is still warming up.")
    time.sleep(60)
print("Wait complete. Querying Cost Explorer now...")

# Query Cost Explorer for the last 4 hours, HOURLY, filtered by the lab tag.
ce = boto3.client(
    "ce",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name="us-east-1",
)
end = datetime.now(timezone.utc)
start = end - timedelta(hours=4)
try:
    ce_resp = ce.get_cost_and_usage(
        TimePeriod={
            "Start": start.strftime("%Y-%m-%dT%H:00:00Z"),
            "End": end.strftime("%Y-%m-%dT%H:00:00Z"),
        },
        Granularity="HOURLY",
        Metrics=["UnblendedCost"],
        Filter={"Tags": {"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}},
    )
    print("Cost Explorer response received.")
    for bucket in ce_resp.get("ResultsByTime", []):
        amt = bucket.get("Total", {}).get("UnblendedCost", {}).get("Amount", "0")
        print(f"  bucket {bucket['TimePeriod']['Start']} -> {bucket['TimePeriod']['End']}: ${amt}")
except ClientError as e:
    print(f"Cost Explorer query failed: {e}")
    print("Re-run this cell; CE can take up to 30 minutes for hourly data.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: cleanup completed clean, post-cleanup 10-min bucket has
# no new charges, tag audit returns zero orphans.

def checkpoint_4(session):
    try:
        if CLEANUP_COMPLETED_AT is None:
            return CheckpointResult(
                status="fail",
                error_reason="Cleanup has not run. Run the cleanup cell at the bottom of this notebook first.",
            )

        # 1. Tag audit: no orphans tagged for this lab remain.
        rg = boto3.client(
            "resourcegroupstaggingapi",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        leftovers = []
        for page in rg.get_paginator("get_resources").paginate(
            TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
        ):
            for item in page.get("ResourceTagMappingList", []):
                leftovers.append(item.get("ResourceARN", ""))
        if leftovers:
            return CheckpointResult(
                status="fail",
                error_reason=f"Tag audit found {len(leftovers)} orphan resources: {leftovers[:3]}",
            )

        # 2. Cost Explorer: most recent hour bucket has no new charges.
        # CE eventual-consistency: accept empty bucket as "no new charges".
        ce = boto3.client(
            "ce",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name="us-east-1",
        )
        end = datetime.now(timezone.utc)
        start = end - timedelta(hours=4)
        try:
            resp = ce.get_cost_and_usage(
                TimePeriod={
                    "Start": start.strftime("%Y-%m-%dT%H:00:00Z"),
                    "End": end.strftime("%Y-%m-%dT%H:00:00Z"),
                },
                Granularity="HOURLY",
                Metrics=["UnblendedCost"],
                Filter={"Tags": {"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}},
            )
        except ClientError as e:
            if e.response["Error"]["Code"] in ("AccessDeniedException",):
                return CheckpointResult(
                    status="fail",
                    error_reason="ce:GetCostAndUsage denied; attach an inline policy and re-run.",
                )
            raise

        # Find the bucket whose Start is the most recent hour at or after
        # CLEANUP_COMPLETED_AT.
        cleanup_hour = CLEANUP_COMPLETED_AT.replace(minute=0, second=0, microsecond=0)
        post_cleanup_amount = 0.0
        for b in resp.get("ResultsByTime", []):
            bstart = datetime.fromisoformat(b["TimePeriod"]["Start"].replace("Z", "+00:00"))
            if bstart >= cleanup_hour:
                try:
                    post_cleanup_amount = max(
                        post_cleanup_amount,
                        float(b.get("Total", {}).get("UnblendedCost", {}).get("Amount", "0") or 0),
                    )
                except (TypeError, ValueError):
                    pass

        # Empty hourly bucket from CE (no data yet) is "no new charges"
        # per the brief; only fail if CE definitively shows > 0.01 USD.
        if post_cleanup_amount > 0.01:
            return CheckpointResult(
                status="fail",
                error_reason=f"Cost Explorer shows ${post_cleanup_amount:.4f} accruing in the post-cleanup hour; cleanup did not fully stop the meter.",
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

Checkpoint 4 is three checks: `CLEANUP_COMPLETED_AT` is set (the cleanup cell at the bottom ran), the tag audit returns zero orphans, and Cost Explorer shows no charges in the post-cleanup hour bucket. The cleanup cell at the bottom sets `CLEANUP_COMPLETED_AT` after `run_cleanup` returns; if you ran the cleanup cell and the checkpoint still fails on the marker, the cleanup raised partway through (look for a red traceback above the canonical summary). The orphan check uses the Resource Groups Tagging API which is the same path the lab uses at setup; if it returns leftovers, the manifest is incomplete (a created resource was not registered) or one of the inline adapter deletes silently failed.

</details>

<details><summary>Hint 2 (direction)</summary>

The recommended student flow is: run the cleanup cell at the bottom of this notebook (which sets `CLEANUP_COMPLETED_AT` and prints the canonical summary), then come back to Task 4, run this Task 4 cell (which waits 10 minutes for CE eventual consistency and queries CE for the lab tag), then run the Checkpoint 4 cell. The Task 4 cell intentionally does NOT call `run_cleanup` itself if `CLEANUP_COMPLETED_AT` is unset; it raises `SystemExit(0)` to tell the student to run the cleanup cell first. That separation keeps the cleanup logic in one place (the bottom cell with the `_LabAdapter`) and keeps Task 4 focused on the CE verification.

</details>

<details><summary>Hint 3 (spoiler)</summary>

There is no `YOUR CODE` to fill in for Task 4. The full solution is: run the cleanup cell at the bottom of the notebook, then run this Task 4 cell, then run the Checkpoint 4 cell.

If Checkpoint 4 fails with `Cleanup has not run`, scroll to the cleanup cell at the bottom of the notebook and run it; it sets `CLEANUP_COMPLETED_AT` after `run_cleanup` returns.

If Checkpoint 4 fails with `Tag audit found N orphan resources`, the cleanup cell ran but at least one resource was not in the manifest or its inline adapter silently failed. The cleanup cell prints WARNING lines for each adapter failure with the exact CLI command to finish the teardown manually; copy/paste those into your shell, then re-run the checkpoint.

If Checkpoint 4 fails with `Cost Explorer shows $... accruing`, Cost Explorer is reporting a charge in the post-cleanup hour. The most likely cause is a Redshift cluster that did not actually delete; run `aws redshift describe-clusters --cluster-identifier labrun-end-to-end-data-platform` from the CLI. If the cluster is in `deleting`, wait 5 more minutes and re-run; if `available`, run `aws redshift delete-cluster --cluster-identifier labrun-end-to-end-data-platform --skip-final-cluster-snapshot`.

</details>


**Wallet check.** Cleanup ran. Cluster and stream meters are stopped. Glue, Firehose, Lambda, EB, SNS, CloudWatch, Lake Formation all back to zero. Cost Explorer is the lagging indicator; the 10-minute wait gives AWS time to publish hourly data. Cumulative session total once the cleanup-bucket settles: $0.75 to $1.50.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation,
# critical-first order per RESOURCE-SAFETY-SPEC Rule 2.
#
# labrun-checks 0.3.0 ships AWS adapters for s3_bucket, iam_role,
# glue_database, and glue_table. Everything else is inlined here:
#   redshift_cluster, kinesis_stream, eventbridge_rule, lambda_function,
#   security_group, glue_workflow, glue_trigger, glue_crawler, glue_job,
#   firehose_delivery_stream, cloudwatch_alarm, sns_topic,
#   sns_subscription, lakeformation_permissions, lakeformation_resource,
#   athena_workgroup.
# When labrun-checks promotes these in a future release, the wrapper can
# be removed and run_cleanup called against the default.

from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter:
    """AwsCleanupAdapter wrapper that adds Lab 12 inline resource types.

    Every inlined type swallows the "already gone" error code so a second
    cleanup invocation (atexit racing the explicit run) is a no-op.
    """

    def __init__(self) -> None:
        self._aws = AwsCleanupAdapter()

    def _client(self, service: str, credentials: dict):
        return boto3.client(
            service,
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
            aws_session_token=credentials.get("aws_session_token"),
            region_name=credentials.get("region", REGION),
        )

    def delete_resource(self, credentials: dict, resource) -> None:
        if resource.type == "redshift_cluster":
            c = self._client("redshift", credentials)
            try:
                c.delete_cluster(ClusterIdentifier=resource.id, SkipFinalClusterSnapshot=True)
            except ClientError as e:
                code_ = e.response["Error"]["Code"]
                if code_ in ("ClusterNotFound", "ClusterNotFoundFault"):
                    return
                if code_ != "InvalidClusterState":
                    raise
            wait_deadline = time.time() + 600
            while time.time() < wait_deadline:
                try:
                    c.describe_clusters(ClusterIdentifier=resource.id)
                except ClientError as e:
                    if e.response["Error"]["Code"] in ("ClusterNotFound", "ClusterNotFoundFault"):
                        return
                    raise
                time.sleep(15)
            return

        if resource.type == "kinesis_stream":
            c = self._client("kinesis", credentials)
            try:
                c.delete_stream(StreamName=resource.id, EnforceConsumerDeletion=True)
            except ClientError as e:
                if e.response["Error"]["Code"] != "ResourceNotFoundException":
                    raise
            wait_deadline = time.time() + 180
            while time.time() < wait_deadline:
                try:
                    c.describe_stream(StreamName=resource.id)
                except ClientError as e:
                    if e.response["Error"]["Code"] == "ResourceNotFoundException":
                        return
                    raise
                time.sleep(5)
            return

        if resource.type == "eventbridge_rule":
            c = self._client("events", credentials)
            try:
                c.remove_targets(Rule=resource.id, Ids=["1"], Force=True)
            except ClientError as e:
                if e.response["Error"]["Code"] not in ("ResourceNotFoundException", "InternalException"):
                    raise
            try:
                c.delete_rule(Name=resource.id, Force=True)
            except ClientError as e:
                if e.response["Error"]["Code"] != "ResourceNotFoundException":
                    raise
            return

        if resource.type == "lambda_function":
            c = self._client("lambda", credentials)
            try:
                c.delete_function(FunctionName=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "ResourceNotFoundException":
                    raise
            return

        if resource.type == "security_group":
            c = self._client("ec2", credentials)
            try:
                resp = c.describe_security_groups(Filters=[{"Name": "group-name", "Values": [resource.id]}])
                for sg in resp.get("SecurityGroups", []):
                    try:
                        c.delete_security_group(GroupId=sg["GroupId"])
                    except ClientError as e:
                        if e.response["Error"]["Code"] != "InvalidGroup.NotFound":
                            raise
            except ClientError as e:
                if e.response["Error"]["Code"] != "InvalidGroup.NotFound":
                    raise
            return

        if resource.type == "glue_workflow":
            c = self._client("glue", credentials)
            try:
                c.delete_workflow(Name=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "EntityNotFoundException":
                    raise
            return

        if resource.type == "glue_trigger":
            c = self._client("glue", credentials)
            try:
                c.delete_trigger(Name=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "EntityNotFoundException":
                    raise
            return

        if resource.type == "glue_crawler":
            c = self._client("glue", credentials)
            try:
                c.delete_crawler(Name=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "EntityNotFoundException":
                    raise
            return

        if resource.type == "glue_job":
            c = self._client("glue", credentials)
            try:
                c.delete_job(JobName=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "EntityNotFoundException":
                    raise
            return

        if resource.type == "firehose_delivery_stream":
            c = self._client("firehose", credentials)
            try:
                c.delete_delivery_stream(DeliveryStreamName=resource.id, AllowForceDelete=True)
            except ClientError as e:
                if e.response["Error"]["Code"] != "ResourceNotFoundException":
                    raise
            wait_deadline = time.time() + 180
            while time.time() < wait_deadline:
                try:
                    c.describe_delivery_stream(DeliveryStreamName=resource.id)
                except ClientError as e:
                    if e.response["Error"]["Code"] == "ResourceNotFoundException":
                        return
                    raise
                time.sleep(5)
            return

        if resource.type == "cloudwatch_alarm":
            c = self._client("cloudwatch", credentials)
            try:
                c.delete_alarms(AlarmNames=[resource.id])
            except ClientError as e:
                if e.response["Error"]["Code"] != "ResourceNotFound":
                    raise
            return

        if resource.type == "sns_subscription":
            c = self._client("sns", credentials)
            topic_arn = f"arn:aws:sns:{REGION}:{ACCOUNT_ID}:{SNS_TOPIC_NAME}"
            try:
                paginator = c.get_paginator("list_subscriptions_by_topic")
                for page in paginator.paginate(TopicArn=topic_arn):
                    for sub in page.get("Subscriptions", []):
                        sub_arn = sub.get("SubscriptionArn", "")
                        if sub_arn and sub_arn != "PendingConfirmation":
                            try:
                                c.unsubscribe(SubscriptionArn=sub_arn)
                            except ClientError:
                                pass
            except ClientError as e:
                if e.response["Error"]["Code"] != "NotFound":
                    pass
            return

        if resource.type == "sns_topic":
            c = self._client("sns", credentials)
            topic_arn = f"arn:aws:sns:{REGION}:{ACCOUNT_ID}:{resource.id}"
            try:
                c.delete_topic(TopicArn=topic_arn)
            except ClientError as e:
                if e.response["Error"]["Code"] != "NotFound":
                    raise
            return

        if resource.type == "lakeformation_permissions":
            c = self._client("lakeformation", credentials)
            for body in LF_GRANT_BODIES:
                try:
                    c.revoke_permissions(**body)
                except ClientError as e:
                    if e.response["Error"]["Code"] not in ("EntityNotFoundException", "AccessDeniedException"):
                        pass
            return

        if resource.type == "lakeformation_resource":
            c = self._client("lakeformation", credentials)
            try:
                c.deregister_resource(ResourceArn=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] not in ("EntityNotFoundException",):
                    pass
            return

        if resource.type == "athena_workgroup":
            c = self._client("athena", credentials)
            try:
                c.delete_work_group(WorkGroup=resource.id, RecursiveDeleteOption=True)
            except ClientError as e:
                if e.response["Error"]["Code"] not in ("InvalidRequestException",):
                    raise
            return

        return self._aws.delete_resource(credentials, resource)

    def describe_resource(self, credentials: dict, resource) -> bool:
        if resource.type == "redshift_cluster":
            c = self._client("redshift", credentials)
            try:
                c.describe_clusters(ClusterIdentifier=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] in ("ClusterNotFound", "ClusterNotFoundFault"):
                    return False
                raise

        if resource.type == "kinesis_stream":
            c = self._client("kinesis", credentials)
            try:
                c.describe_stream(StreamName=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise

        if resource.type == "eventbridge_rule":
            c = self._client("events", credentials)
            try:
                c.describe_rule(Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise

        if resource.type == "lambda_function":
            c = self._client("lambda", credentials)
            try:
                c.get_function(FunctionName=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise

        if resource.type == "security_group":
            c = self._client("ec2", credentials)
            try:
                resp = c.describe_security_groups(Filters=[{"Name": "group-name", "Values": [resource.id]}])
                return bool(resp.get("SecurityGroups", []))
            except ClientError as e:
                if e.response["Error"]["Code"] == "InvalidGroup.NotFound":
                    return False
                raise

        if resource.type == "glue_workflow":
            c = self._client("glue", credentials)
            try:
                c.get_workflow(Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return False
                raise

        if resource.type == "glue_trigger":
            c = self._client("glue", credentials)
            try:
                c.get_trigger(Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return False
                raise

        if resource.type == "glue_crawler":
            c = self._client("glue", credentials)
            try:
                c.get_crawler(Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return False
                raise

        if resource.type == "glue_job":
            c = self._client("glue", credentials)
            try:
                c.get_job(JobName=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return False
                raise

        if resource.type == "firehose_delivery_stream":
            c = self._client("firehose", credentials)
            try:
                c.describe_delivery_stream(DeliveryStreamName=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise

        if resource.type == "cloudwatch_alarm":
            c = self._client("cloudwatch", credentials)
            try:
                resp = c.describe_alarms(AlarmNames=[resource.id])
                return bool(resp.get("MetricAlarms", []))
            except ClientError:
                return False

        if resource.type == "sns_subscription":
            return False  # adapter resolves all subs in delete; describe is best-effort

        if resource.type == "sns_topic":
            c = self._client("sns", credentials)
            topic_arn = f"arn:aws:sns:{REGION}:{ACCOUNT_ID}:{resource.id}"
            try:
                c.get_topic_attributes(TopicArn=topic_arn)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "NotFound":
                    return False
                raise

        if resource.type == "lakeformation_permissions":
            return False  # adapter revokes from LF_GRANT_BODIES on every call

        if resource.type == "lakeformation_resource":
            c = self._client("lakeformation", credentials)
            try:
                c.describe_resource(ResourceArn=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] in ("EntityNotFoundException",):
                    return False
                raise

        if resource.type == "athena_workgroup":
            c = self._client("athena", credentials)
            try:
                c.get_work_group(WorkGroup=resource.id)
                return True
            except ClientError:
                return False

        return self._aws.describe_resource(credentials, resource)

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        return self._aws.tag_scan(credentials, lab_slug, region)


# Empty the S3 bucket before the s3_bucket adapter tears it down.
print("Emptying the S3 bucket before teardown...")
s3_clean = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    paginator = s3_clean.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3_clean.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing to cleanup.")

# Detach inline policies on every IAM role we created so the iam_role
# adapter's plain delete-role does not block.
iam_clean = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
for role_name in (
    FIREHOSE_ROLE_NAME, GLUE_ROLE_NAME, COPY_ROLE_NAME,
    ANALYTICS_ROLE_NAME, SAFETY_NET_ROLE_NAME,
):
    try:
        listed = iam_clean.list_role_policies(RoleName=role_name)
        for policy_name in listed.get("PolicyNames", []):
            iam_clean.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
    except ClientError as e:
        if e.response["Error"]["Code"] != "NoSuchEntity":
            print(f"Inline policy detach for {role_name}: {e}. Continuing.")

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if getattr(r, "critical", False))
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_destroyed = critical_total - sum(
    1 for r in CLEANUP_MANIFEST if getattr(r, "critical", False) and r.id in failed_ids
)
standard_destroyed = standard_total - sum(
    1 for r in CLEANUP_MANIFEST if not getattr(r, "critical", False) and r.id in failed_ids
)
failed_count = len(failed_ids)

CLEANUP_COMPLETED_AT = datetime.now(timezone.utc)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")
print()
print(f"Cleanup completion timestamp recorded: {CLEANUP_COMPLETED_AT.isoformat()}")
print("Now run the Task 4 cell to wait 10 minutes and call Cost Explorer.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: $0.75 to $1.50.** The capstone is the most expensive lab in the cert. Redshift `dc2.large` at $0.25 per node-hour for the 75-minute session is about $0.31; Kinesis at $0.015 per shard-hour for the same window is $0.02; Glue ETL DPU-hours across crawler + 2 jobs + debugging put $0.30 to $0.80 on the meter; Firehose at $0.029 per GB ingested rounds to under a cent for the canary volume; CloudWatch, EventBridge, SNS, Lake Formation, and Lambda are all free at this volume. If you walked away and the 3-hour safety-net Lambda fired, add another $0.45 of Redshift and $0.05 of Kinesis for a worst case of about $1.50. The cleanup cell above deleted the cluster, the stream, the safety-net pair, the LF grants and registration, the alarm + EB rules + SNS chain, the Glue triggers + workflow + jobs + crawler + table + database, the Firehose stream, all five IAM roles, the security group, and the S3 bucket. Cost Explorer should show zero new charges within the next 10 minutes.


## These are not graded. They are for you.

Three questions worth sitting with before the exam.

1. The workflow ships with three triggers. The start trigger is `ON_DEMAND`; the transform and load triggers are `CONDITIONAL`. The brief mentions a fourth, `SCHEDULED`, trigger that the lab does not enable. Walk through the trade-offs of adding a scheduled trigger to the same workflow in production: backfill vs. fresh-data ingestion, idempotency of the load job's COPY (Lab 06 used `TRUNCATE`; this lab uses Parquet partition-overwrite with bookmark-resume), the cost of a 24-hour cluster vs. spinning up Redshift Serverless per run. This maps to DEA-C01 Domain 1 expectations on workflow orchestration patterns.

2. Checkpoint 4 is graded on Cost Explorer showing no charges 10 minutes post-cleanup. Walk through the failure modes of that check: CE eventual-consistency lag (the brief says up to 30 minutes for hourly); a Redshift cluster stuck in `deleting` for longer than 10 minutes; an S3 bucket that did not empty because the Parquet partition under `curated/year=2024/month=11/` was not enumerated by the paginator; the Cost Explorer filter not matching because the resource was tagged at the wrong key. Which of these would the validator's "empty bucket means no charges" rule paper over, and which would surface as a real failure?

3. The safety-net Lambda is tag-scoped: it confirms `labrun:lab-slug == "end-to-end-data-platform"` on each resource before any delete API. Walk through the failure mode where a student's session has TWO Lab 12 attempts running in parallel (one Jupyter kernel doing setup, another kernel doing cleanup). What invariant does the tag-scope provide? What does NOT provide? Hint: the EB schedule rule's one-shot cron is independent per session; if both sessions provision and one finishes early, the other's safety net still has a 3-hour clock. This is the kind of concurrency question DEA-C01 likes to ask about operational reliability.

## Exam-Style Practice

**Q1.** A data engineering team uses CloudWatch alarms to detect silent Glue job failures, with the alarm's `AlarmActions` set to an SNS topic and an EventBridge rule on `aws.cloudwatch CloudWatch Alarm State Change` events also targeting the same SNS topic. When the alarm flips to ALARM, the team sees TWO notifications instead of one. Which option correctly explains why?

A) The alarm is misconfigured; `AlarmActions` and EventBridge fan-out are mutually exclusive and should never both be wired to the same topic.

B) Both the alarm's direct `AlarmActions` and the EventBridge rule on alarm state change publish to SNS independently; this is the expected double-fan-out and the team should pick one path.

C) SNS deduplicates by message ID; the duplicate is a transient AWS bug that resolves within 24 hours.

D) The CloudWatch alarm is in a different region than the SNS topic; cross-region alarm actions trigger a retry that publishes twice.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. The two paths are perfectly valid and commonly combined; nothing in CloudWatch or EventBridge prevents both from being wired to the same topic.
- B) Right. The `AlarmActions` field on `put_metric_alarm` is one publish path; the EventBridge rule on `aws.cloudwatch CloudWatch Alarm State Change` is a second, independent path. Both fire when the alarm flips, and both publish to whatever target they are wired to. To avoid duplicates, pick one: typically `AlarmActions` for synchronous notifications and EventBridge for routing to downstream automation like Lambda or Step Functions.
- C) Wrong. SNS does not deduplicate by message ID; standard topics deliver every publish at least once.
- D) Wrong. CloudWatch alarms and SNS topics are region-scoped, but cross-region alarm actions are not even allowed; if they were, the failure would be a permissions error, not a retry-then-double-publish.

</details>

**Q2.** A data engineer needs to grant a team_a analytics role SELECT on a Glue table `curated_orders` that has columns `order_id`, `customer_email`, `event_id`, `amount_cents`, `currency`, and `created_at`, with the analyst blocked from reading `customer_email`. The engineer uses Lake Formation's `grant_permissions` API. Which `Resource` shape is the canonical and most maintainable form?

A) `Resource.Table` (database + table name) with `Permissions=["SELECT"]`; column-level enforcement is automatic.

B) `Resource.TableWithColumns` with `ColumnNames=["order_id", "event_id", "amount_cents", "currency", "created_at"]` and `Permissions=["SELECT"]`.

C) `Resource.TableWithColumns` with `ColumnWildcard.ExcludedColumnNames=["customer_email"]` and `Permissions=["SELECT"]`.

D) `Resource.DataLocation` on the S3 prefix; column-level filtering happens at the Athena workgroup level.

<details><summary>Show answer</summary>

**C**.

- A) Wrong. `Resource.Table` grants SELECT on all columns, including `customer_email`. Lake Formation does not infer column-level boundaries from the table grant.
- B) Right behavior, less maintainable. The explicit `ColumnNames` form does grant SELECT on the five non-PII columns and blocks `customer_email`. The downside is every new column added to the table requires updating the grant, or the analyst silently loses access.
- C) Most maintainable. `ColumnWildcard.ExcludedColumnNames=["customer_email"]` grants SELECT on every column EXCEPT `customer_email`, so new non-PII columns added to the table are automatically accessible without re-granting. This is the canonical pattern for "block these columns, allow the rest."
- D) Wrong. `Resource.DataLocation` grants the ability to register a location, not row or column SELECT. Column filtering is a Lake Formation table-grant concern, not an Athena workgroup concern.

</details>

**Q3.** A production pipeline uses a Kinesis Data Stream as the source for a Firehose delivery stream that writes to S3. The pipeline runs idle overnight and the on-call engineer sees that AWS still charges shard-hours on the stream. Which option correctly explains why and what stops the bill?

A) The Firehose delivery stream is the billable component; pausing Firehose stops the shard-hour charges on the upstream Kinesis stream.

B) Kinesis Data Streams bill per shard-hour from create until delete, independent of whether records are produced; the only way to stop shard-hour charges is to delete the stream.

C) The shard-hour billing is for the Firehose consumer; setting `EnhancedFanOut=False` on Firehose stops the charge.

D) Shard-hour charges only apply to producers; the consumer is free. The bill must be coming from CloudWatch metric storage on the stream.

<details><summary>Show answer</summary>

**B**.

- B) Right. Kinesis Data Streams bill at $0.015 per shard-hour from the moment `create_stream` returns until `delete_stream` is called. Idle streams still incur shard-hours; the only way to stop the bill is to delete the stream. RESOURCE-SAFETY-SPEC flags this as a critical resource for exactly this reason.
- A) Wrong. Pausing Firehose has no effect on the upstream Kinesis stream's shard-hour bill. The two services bill independently.
- C) Wrong. There is no `EnhancedFanOut=False` flag on Firehose; enhanced fan-out is a Kinesis consumer feature, and Firehose-as-consumer uses the shared throughput path. Either way, this does not stop shard-hour billing.
- D) Wrong. Shard-hour billing applies to the stream itself regardless of producer or consumer activity. CloudWatch metric storage is a separate, much smaller line item.

</details>
